<a href="https://colab.research.google.com/github/JorgeZorrilla/Crash-GeoNN/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initialization

In [ ]:
# Colab setup: install PyTorch Geometric wheels matching your Torch/CUDA
import torch, sys, os, platform, subprocess, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# This magic line pulls the right wheels for your torch+cuda combo
torch_ver = torch.__version__.split('+')[0]
cuda_tag = (torch.version.cuda or 'cpu').replace('.', '')
index_url = f"https://data.pyg.org/whl/torch-{torch_ver}%2B{cuda_tag}.html"

!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric \
  -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

!pip install pyvista imageio-ffmpeg
!pip install optuna

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)



Torch: 2.8.0+cu126 | CUDA: 12.6
Device: cuda


Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # autoriza y usa rutas como '/content/drive/MyDrive/...'


Mounted at /content/drive


Import dependencies

In [ ]:
import os, math, random, numpy as np, time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Dict, List, Tuple, Sequence
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GraphSAGE
from torch_cluster import radius_graph
from tqdm.auto import tqdm
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

## Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    seed = torch.initial_seed() % 2**31
    np.random.seed(seed + worker_id); random.seed(seed + worker_id)

def check_sim(steps: List[Data], sid: int, max_print_edges=5):
    assert isinstance(steps, (list, tuple)) and len(steps) >= 1, f"[sim {sid}] bad list"
    N = steps[0].x.shape[0]
    E = steps[0].edge_index.shape[1]
    pos0 = getattr(steps[0], 'pos0', None)
    edge_index0 = steps[0].edge_index
    issues = []
    for t, g in enumerate(steps):
        if not isinstance(g, Data): issues.append(f"step {t} not Data"); continue
        if g.x.dim()!=2 or g.y.dim()!=2: issues.append(f"step {t} x/y dim !=2")
        if g.x.shape[0]!=N or g.y.shape[0]!=N: issues.append(f"step {t} N mismatch")
        if g.edge_index.shape[0]!=2 or g.edge_index.shape[1]!=E: issues.append(f"step {t} ei shape mismatch")
        if not torch.equal(g.edge_index, edge_index0): issues.append(f"step {t} ei differs")
        if int(g.edge_index.max()) >= N: issues.append(f"step {t} ei out of range")
        if not torch.isfinite(g.x).all() or not torch.isfinite(g.y).all(): issues.append(f"step {t} NaN/Inf in x/y")
        if hasattr(g, "edge_attr"):
            if not torch.isfinite(g.edge_attr).all(): issues.append(f"step {t} NaN/Inf in edge_attr")
    ei = edge_index0.t().tolist()
    undirected = all(([j,i] in ei) for i,j in ei[:max_print_edges])
    unique_pairs = set(tuple(sorted(e)) for e in ei)
    dup = (len(unique_pairs) * 2 != len(ei))
    print(f"[sim {sid}] N={N} E={E} undirected? {undirected} duplicates? {dup}")
    if issues: print("  Issues:", "; ".join(issues))

def transform_edge_attr(edge_attr: torch.Tensor, edge_scaler):
    if edge_attr is None or edge_scaler is None:
        return edge_attr
    em, es = edge_scaler
    return (edge_attr - em) / es

def drop_features_db(db: List[List[Data]], drop_idx_x: List[int], drop_idx_y: List[int]):
    """
    Elimina atributos (columnas) de x (y opcionalmente de y) en TODA la base de datos.

    Args:
        db: List[List[Data]]  -> base de datos completa
        drop_idx: lista de índices de columnas a eliminar
    """
    if not drop_idx_x and not drop_idx_y:
        return db  # nada que hacer

    drop_idx_x = sorted(set(drop_idx_x))
    drop_idx_y = sorted(set(drop_idx_y))

    for sim in db:
        for g in sim:
            # --- X ---
            if hasattr(g, "x") and g.x is not None:
                keep_x = [i for i in range(g.x.size(1)) if i not in drop_idx_x]
                g.x = g.x[:, keep_x]

            # --- Y (opcional) ---
            if hasattr(g, "y") and g.y is not None:
              keep_y = [i for i in range(g.y.size(1)) if i not in drop_idx_y]
              g.y = g.y[:, keep_y]


    return db



def tensor3d_to_dfs(arr3d, feature_names=None):
    """
    arr3d: (T, N, C) en numpy o torch
    feature_names: lista de nombres de longitud C (opcional)
    """
    # -> numpy
    if isinstance(arr3d, torch.Tensor):
        A = arr3d.detach().cpu().numpy()
    else:
        A = np.asarray(arr3d)
    T, N, C = A.shape

    # Nombres de columnas
    cols = feature_names if feature_names is not None else [f"f{i}" for i in range(C)]

    # ---- Wide: index=(t,node), columns=features ----
    idx = pd.MultiIndex.from_product([range(T), range(N)], names=["t", "node"])
    df_wide = pd.DataFrame(A.reshape(T*N, C), index=idx, columns=cols)

    # ---- Long: tidy ----
    t_idx   = np.repeat(np.arange(T), N*C)
    node_idx= np.tile(np.repeat(np.arange(N), C), T)
    feat_idx= np.tile(np.arange(C), T*N)
    feat    = np.array(cols)[feat_idx]
    values  = A.reshape(-1)
    df_long = pd.DataFrame({"t": t_idx, "node": node_idx, "feature": feat, "value": values})

    return df_long, df_wide

## Load database functions

In [ ]:
def load_database(path_pt: str) -> List[List[Data]]:
    # print("Loading DB from:", path_pt)
    db = torch.load(path_pt, map_location="cpu", weights_only=False)
    return db

def load_database_dir(dir_path: str, step = 1) -> List[List[Data]]:
    print("Loading DB from:", dir_path)
    db = []
    graphs = os.listdir(dir_path)
    if graphs:
      print(f"Found {len(graphs)} graphs")
      for i in tqdm(range(0, len(graphs), step)):
        path = os.path.join(dir_path, graphs[i])
        db.append(load_database(path))
      for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
      lens = [len(s) for s in db]
    return db
def split_simulations(all_sim_ids, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.default_rng(seed); ids = np.array(all_sim_ids); rng.shuffle(ids)
    n = len(ids); n_tr = int(n*train_ratio); n_va = int(n*val_ratio)
    return ids[:n_tr].tolist(), ids[n_tr:n_tr+n_va].tolist(), ids[n_tr+n_va:].tolist()

def build_split_from_db(db: List[List[Data]], sim_ids: List[int], min_time_step: int = 0):
    graphs, sim_static = [], {}
    for sid in sim_ids:
        steps_all = db[sid]
        assert len(steps_all) >= 1, f"Simulation {sid} empty."

        # Si no hay suficientes pasos, saltamos la simulación
        if len(steps_all) <= min_time_step:
            print(f"[WARN] sim {sid} skipped: len(steps)={len(steps_all)} <= min_t={min_time_step}")
            continue

        # Filtrado por timestep
        steps = steps_all[min_time_step:]                        # Data_t(min_time_step) .. Data_t(T-2)
        edge_index = steps[0].edge_index
        pos0 = getattr(steps[0], 'pos0', None)
        simulation_id = getattr(steps[0], 'simulation_id', None)
        bc_mask = getattr(steps[0], 'bc_mask', None)
        rigid_mask = getattr(steps[0], 'rigid_mask', None)
        timestep_index = getattr(steps[0], 't_idx', None)
        # fixed_idx = getattr(steps[0], 'fixed_idx', None)
        edge_attr = getattr(steps[0], 'edge_attr', None)

        # K = len(steps) = (T-1 - min_time_step)
        # Estados efectivos: ΔX_{min_time_step} .. ΔX_T  -> T_eff = K + 1
        T_eff = len(steps) + 1

        # Ground-truth a partir de min_time_step: ΔX_{min_time_step+1 .. T}
        y_real = torch.stack([d.y for d in steps], dim=0)  # (T_eff-1, N, N_features)

        # Estado inicial para rollout: ΔX_{min_t} (ojo: sin normalizar)
        x0 = steps_all[min_time_step].x.detach().clone()

        # Añadimos los Data filtrados al conjunto de entrenamiento/val/test
        graphs.extend(steps)

        sim_static[sid] = {
            'simulation_id' : simulation_id,
            'bc_mask' : bc_mask,
            'rigid_mask' : rigid_mask,
            'timestep_index' : timestep_index,
            # 'fixed_idx' : fixed_idx,
            'edge_index': edge_index,
            'edge_attr' : edge_attr,     # OJO: aún sin escalar aquí
            'pos0': pos0,
            'T_eff': T_eff, # Number of effective timesteps
            'y_real': y_real,
            'x0': x0           # punto de partida del rollout
        }

    return graphs, sim_static

## Normalization functions

In [ ]:
import torch
from typing import Dict, List, Tuple, Sequence, Iterable, Optional
from torch_geometric.data import Data

# ---------- helpers ----------

def _flatten_graphs(graphs: Iterable):
    """Acepta [Data,...] o [[Data,...], [Data,...], ...]"""
    graphs = list(graphs)
    if len(graphs) == 0:
        return []
    if all(hasattr(g, "__iter__") and not hasattr(g, "x") for g in graphs):
        out = []
        for sub in graphs:
            out.extend(sub)
        return out
    return graphs

def _gather_cols(graphs: List[Data], attr: str, idxs: Sequence[int]) -> torch.Tensor:
    """Concatena a lo largo de la dim de filas, recogiendo columnas 'idxs' de la última dim."""
    xs = []
    for g in graphs:
        if not hasattr(g, attr): continue
        X = getattr(g, attr)
        if X is None or X.numel() == 0: continue
        # Selección de columnas en la última dimensión, soporta (N,C) o (T,N,C)
        Xc = X[..., idxs]
        # Colapsa todo menos canales -> (M, Csel)
        Xc = Xc.reshape(-1, Xc.shape[-1]).float()
        xs.append(Xc)
    if not xs:
        raise ValueError(f"No hay muestras para '{attr}' en las columnas {list(idxs)}")
    return torch.cat(xs, dim=0)  # (M, Csel)

def _check_indices(name: str, F: int, idxs: Sequence[int]):
    if idxs is None:
        raise ValueError(f"{name}: índices None")
    bad = [i for i in idxs if i < 0 or i >= F]
    if bad:
        raise IndexError(f"{name}: índices fuera de rango {bad} para F={F}")
    if len(set(idxs)) != len(idxs):
        print(f"[WARN] {name}: índices duplicados detectados -> {idxs}")

def _nan_aware_mean_std(X: torch.Tensor, eps: float):
    # Ignora NaN/Inf sustituyéndolos por valores finitos al calcular stats
    mask = torch.isfinite(X)
    if mask.all():
        m = X.mean(0, keepdim=True)
        s = X.std(0, keepdim=True).clamp_min(eps)
    else:
        Xm = torch.where(mask, X, torch.nan)
        m = torch.nanmean(Xm, dim=0, keepdim=True)
        # nanstd no está en todas las versiones; implementamos a mano
        diff2 = (Xm - m)**2
        v = torch.nanmean(diff2, dim=0, keepdim=True)
        s = v.sqrt().clamp_min(eps)
    return m, s

# ---------- fits ----------

@torch.no_grad()
def fit_scaler(
    graphs: List[Data],
    affected_index_x: Sequence[int],
    affected_index_y: Sequence[int],
    eps: float = 1e-8,
    verbose: bool = False
):
    graphs = _flatten_graphs(graphs)
    if len(graphs) == 0:
        raise ValueError("fit_scaler: graphs vacío")

    # Dimensiones base
    if not hasattr(graphs[0], "x") or graphs[0].x is None:
        raise ValueError("fit_scaler: graphs[0].x es None")
    if not hasattr(graphs[0], "y") or graphs[0].y is None:
        raise ValueError("fit_scaler: graphs[0].y es None")

    len_x = graphs[0].x.shape[-1]
    len_y = graphs[0].y.shape[-1]
    _check_indices("X", len_x, affected_index_x)
    _check_indices("Y", len_y, affected_index_y)

    # X: solo columnas afectadas
    Xc = _gather_cols(graphs, "x", affected_index_x)
    xm_c, xs_c = _nan_aware_mean_std(Xc, eps)

    xm = torch.zeros(1, len_x, dtype=xm_c.dtype, device=xm_c.device)
    xs = torch.ones(1,  len_x, dtype=xs_c.dtype, device=xs_c.device)
    xm[:, affected_index_x] = xm_c
    xs[:, affected_index_x] = xs_c

    # Y: solo columnas afectadas
    Yc = _gather_cols(graphs, "y", affected_index_y)
    ym_c, ys_c = _nan_aware_mean_std(Yc, eps)

    ym = torch.zeros(1, len_y, dtype=ym_c.dtype, device=ym_c.device)
    ys = torch.ones(1,  len_y, dtype=ys_c.dtype, device=ys_c.device)
    ym[:, affected_index_y] = ym_c
    ys[:, affected_index_y] = ys_c

    if verbose:
        print(f"[fit_scaler] X: muestras={Xc.shape[0]}, cols_afectadas={len(affected_index_x)}, "
              f"std[min,max]=({xs_c.min().item():.3g},{xs_c.max().item():.3g})")
        print(f"[fit_scaler] Y: muestras={Yc.shape[0]}, cols_afectadas={len(affected_index_y)}, "
              f"std[min,max]=({ys_c.min().item():.3g},{ys_c.max().item():.3g})")

    return (xm, xs), (ym, ys)

@torch.no_grad()
def fit_pos_scaler(static_dict, eps: float = 1e-8, verbose: bool = False):
    P = []
    for sid, info in static_dict.items():
        pos0 = info.get('pos0', None)
        if pos0 is None: continue
        P.append(pos0)  # (N, 3) o (N, D)
    if not P:
        raise AssertionError("No hay pos0 en train_static para ajustar pos_scaler")
    P = torch.cat(P, dim=0).float()
    pm, ps = _nan_aware_mean_std(P, eps)
    if verbose:
        print(f"[fit_pos_scaler] muestras={P.shape[0]}, C={P.shape[1]}, std[min,max]=({ps.min().item():.3g},{ps.max().item():.3g})")
    return (pm, ps)

@torch.no_grad()
def fit_edge_attr_scaler(graphs: List[Data], eps: float = 1e-8, verbose: bool = False):
    E_list = [g.edge_attr for g in _flatten_graphs(graphs)
              if hasattr(g, "edge_attr") and g.edge_attr is not None and g.edge_attr.numel() > 0]
    if not E_list:
        if verbose: print("[fit_edge_attr_scaler] no hay edge_attr; devuelvo None")
        return None
    E = torch.cat([e.reshape(-1, e.shape[-1]).float() for e in E_list], dim=0)
    em, es = _nan_aware_mean_std(E, eps)
    if verbose:
        print(f"[fit_edge_attr_scaler] muestras={E.shape[0]}, C={E.shape[1]}, std[min,max]=({es.min().item():.3g},{es.max().item():.3g})")
    return (em, es)

@torch.no_grad()
def fit_delta_scaler(graphs, dyn_idx_x, dyn_idx_y, eps: float = 1e-8, verbose: bool = False):
    dyn_idx_x = list(dyn_idx_x); dyn_idx_y = list(dyn_idx_y)
    if len(dyn_idx_x) != len(dyn_idx_y):
        raise ValueError("fit_delta_scaler: dyn_idx_x y dyn_idx_y deben tener la misma longitud (mapeo 1:1)")

    deltas = []
    for g in _flatten_graphs(graphs):
        x_dyn = g.x[..., dyn_idx_x].reshape(-1, len(dyn_idx_x)).float()
        y_dyn = g.y[..., dyn_idx_y].reshape(-1, len(dyn_idx_y)).float()
        deltas.append(y_dyn - x_dyn)  # Δ_phys
    D = torch.cat(deltas, dim=0)
    dm, ds = _nan_aware_mean_std(D, eps)
    if verbose:
        print(f"[fit_delta_scaler] muestras={D.shape[0]}, C={D.shape[1]}, std[min,max]=({ds.min().item():.3g},{ds.max().item():.3g})")
    return (dm, ds)

@torch.no_grad()
def fit_edge_geom_scaler(train_static: dict, eps: float = 1e-8, verbose: bool = False):
    rels = []
    for sid, info in train_static.items():
        pos0 = info['pos0'].float()
        ei = info['edge_index']
        s, d = ei[0], ei[1]
        rel = pos0[s] - pos0[d]                 # (E,3)
        dist = rel.norm(dim=-1, keepdim=True)   # (E,1)
        rels.append(torch.cat([rel, dist], dim=1))  # (E,4)
    A = torch.cat(rels, dim=0)                  # (sumE,4)
    em, es = _nan_aware_mean_std(A, eps)
    if verbose:
        print(f"[fit_edge_geom_scaler] muestras={A.shape[0]}, C={A.shape[1]}, std[min,max]=({es.min().item():.3g},{es.max().item():.3g})")
    return (em, es)

# ---------- apply ----------

@torch.no_grad()
def apply_scaler(graphs: List[Data], x_scaler, y_scaler, verbose: bool = False):
    xm, xs = x_scaler
    ym, ys = y_scaler
    graphs = _flatten_graphs(graphs)
    # Resumen opcional
    cnt = 0
    for g in graphs:
        if hasattr(g, "x") and g.x is not None:
            dev = g.x.device
            g.x = (g.x - xm.to(dev)) / xs.to(dev)
        if hasattr(g, "y") and g.y is not None:
            dev = g.y.device
            g.y = (g.y - ym.to(dev)) / ys.to(dev)
        cnt += 1
    if verbose:
        # pequeño sanity check agregado (sobre el primer grafo con datos)
        gx = next((gg for gg in graphs if hasattr(gg,"x") and gg.x is not None), None)
        gy = next((gg for gg in graphs if hasattr(gg,"y") and gg.y is not None), None)
        if gx is not None:
            X = gx.x.reshape(-1, gx.x.shape[-1])
            print(f"[apply_scaler] ejemplo X: mean(abs)={X.mean(0).abs().mean().item():.3g}, std(mean)={X.std(0).mean().item():.3g}")
        if gy is not None:
            Y = gy.y.reshape(-1, gy.y.shape[-1])
            print(f"[apply_scaler] ejemplo Y: mean(abs)={Y.mean(0).abs().mean().item():.3g}, std(mean)={Y.std(0).mean().item():.3g}")
        print(f"[apply_scaler] grafos procesados={cnt}")

@torch.no_grad()
def apply_edge_attr_scaler(graphs: List[Data], scaler, verbose: bool = False):
    if scaler is None:
        if verbose: print("[apply_edge_attr_scaler] scaler=None (omitido)")
        return
    em, es = scaler
    graphs = _flatten_graphs(graphs)
    cnt = 0
    for g in graphs:
        if hasattr(g, "edge_attr") and g.edge_attr is not None and g.edge_attr.numel() > 0:
            g.edge_attr = (g.edge_attr - em.to(g.edge_attr)) / es.to(g.edge_attr)
            cnt += 1
    if verbose:
        print(f"[apply_edge_attr_scaler] grafos con edge_attr normalizado={cnt}")

# ---------- sanity checks opcionales ----------

@torch.no_grad()
def sanity_check_attr(graphs: List[Data], attr: str, idxs: Optional[Sequence[int]] = None, k: int = 8):
    """Imprime medias y std por columna (muestras concatenadas) para verificar ~N(0,1)."""
    graphs = _flatten_graphs(graphs)
    if idxs is None:
        F = getattr(graphs[0], attr).shape[-1]
        idxs = list(range(F))
    X = _gather_cols(graphs, attr, idxs)
    m = X.mean(0); s = X.std(0)
    print(f"[sanity_check_{attr}] M={X.shape[0]}, C={X.shape[1]}, "
          f"|mean|_avg={m.abs().mean().item():.3g}, std_avg={s.mean().item():.3g}, "
          f"std[min,max]=({s.min().item():.3g},{s.max().item():.3g})")
    # Mostrar primeras k columnas como muestra:
    kk = min(k, X.shape[1])
    print(f"[sanity_check_{attr}] primeras {kk} cols -> mean={m[:kk].tolist()}, std={s[:kk].tolist()}")


## Models

In [ ]:
from torch_geometric.nn import GINEConv, BatchNorm, LayerNorm, GraphNorm

class ImpactGNN(nn.Module):
    def __init__(self, in_ch=3, hidden=128, out_ch=3, layers=3):
        super().__init__()
        self.gnn = GraphSAGE(in_channels=in_ch, hidden_channels=hidden, num_layers=layers)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, out_ch))
    def forward(self, x, edge_index, edge_attr=None):
        h = self.gnn(x, edge_index)
        return self.head(h)

class ImpactGNN_Edge(nn.Module):
    def __init__(self, in_ch=3, edge_attr_dim=4, hidden=128, out_ch=3, layers=3, dropout=0.1):
        super().__init__()
        convs, norms = [], []
        for l in range(layers):
            mlp = nn.Sequential(
                nn.Linear(hidden if l>0 else in_ch, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )
            convs.append(GINEConv(mlp, edge_dim=edge_attr_dim))
            norms.append(BatchNorm(hidden))
        self.convs = nn.ModuleList(convs)
        self.norms = nn.ModuleList(norms)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_ch))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = x
        for conv, bn in zip(self.convs, self.norms):
            h = conv(h, edge_index, edge_attr)
            h = bn(h); h = F.relu(h); h = self.dropout(h)
        return self.head(h)
class ImpactGNN_Edge_v2(nn.Module):
    def __init__(self, in_ch=3, edge_attr_dim=4, hidden=128, out_ch=3, layers=3, dropout=0.1):
        super().__init__()
        self.edge_enc = nn.Sequential(
            nn.Linear(edge_attr_dim, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden)
        )
        convs, norms = [], []
        for l in range(layers):
            mlp = nn.Sequential(
                nn.Linear(hidden if l>0 else in_ch, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )
            # ¡Ojo! ahora pasaremos edge_feat ya en 'hidden'
            convs.append(GINEConv(mlp, edge_dim=hidden))
            norms.append(BatchNorm(hidden))
        self.convs = nn.ModuleList(convs)
        self.norms = nn.ModuleList(norms)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_ch))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = x
        e = self.edge_enc(edge_attr) if edge_attr is not None else None
        for conv, bn in zip(self.convs, self.norms):
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = self.dropout(h)
        return self.head(h)

class ImpactGNN_EdgeGRU(nn.Module):
    """
    Edge→Node con residuales + estado oculto por nodo (GRU).
    - edge_update: e' = e + φ_e([x_i, x_j, e])
    - node_update: h' = GRU(msg(x, e'), h)  con msg = GINEConv sobre e'
    La cabeza predice Δ (normalizado) a partir de h'.
    """
    def __init__(self, in_ch=3, edge_attr_dim=4, hidden=128, out_ch=3, layers=3, dropout=0.1):
        super().__init__()
        self.hidden = hidden
        self.dropout = nn.Dropout(dropout)

        # Proyección de entrada de nodos a 'hidden'
        self.x_in = nn.Linear(in_ch, hidden)

        # Codificador de atributos de arista (estáticos + dinámicos)
        self.edge_enc = nn.Sequential(
            nn.Linear(edge_attr_dim, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden)
        )

        # Bloques Edge→Node (residuales)
        self.edge_upd = nn.ModuleList([
            nn.Sequential(
                nn.Linear(2*hidden + hidden, hidden),  # [x_i, x_j, e] en espacio hidden
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            ) for _ in range(layers)
        ])
        self.edge_norm = nn.ModuleList([LayerNorm(hidden) for _ in range(layers)])

        self.node_mlps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            ) for _ in range(layers)
        ])
        # Usamos GINEConv como agregador de mensajes con e' ya en 'hidden'
        from torch_geometric.nn import GINEConv, LayerNorm as PygLayerNorm, BatchNorm
        self.convs = nn.ModuleList([
            GINEConv(self.node_mlps[l], edge_dim=hidden) for l in range(layers)
        ])
        self.node_norm = nn.ModuleList([BatchNorm(hidden) for _ in range(layers)])

        # Estado recurrente por nodo
        self.gru = nn.GRUCell(hidden, hidden)

        # Cabeza de predicción (Δ normalizado)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_ch)
        )

    def init_hidden(self, x: torch.Tensor):
        # x: (N, in_ch) -> (N, hidden)
        with torch.no_grad():
            return torch.tanh(self.x_in(x))

    def forward(self, x, edge_index, edge_attr, h_prev=None):
        """
        x: (N, in_ch)  (ya normalizado según tu pipeline)
        edge_attr: (E, edge_attr_dim)  (estáticos + dinámicos normalizados)
        h_prev: (N, hidden) o None
        return: delta_norm (N, out_ch), h_new (N, hidden)
        """
        N = x.size(0)
        h = self.init_hidden(x) if h_prev is None else h_prev

        xh = self.x_in(x)  # (N, hidden)
        e = self.edge_enc(edge_attr) if edge_attr is not None else None

        src, dst = edge_index  # (E,)

        for l in range(len(self.convs)):
            # --- Edge update (residual) ---
            if e is None:
                # si no hay edge_attr, crea dummy zeros
                e = torch.zeros(src.numel(), self.hidden, device=x.device, dtype=x.dtype)
            xi, xj = xh[src], xh[dst]                 # (E, hidden), (E, hidden)
            e_msg = self.edge_upd[l](torch.cat([xi, xj, e], dim=-1))
            e = self.edge_norm[l](e + e_msg)         # residual

            # --- Node message passing ---
            x_msg = self.convs[l](xh, edge_index, e) # (N, hidden)
            x_msg = self.node_norm[l](x_msg)
            x_msg = F.relu(x_msg)
            x_msg = self.dropout(x_msg)

            # --- Recurrent update (GRU por nodo) ---
            h = self.gru(x_msg, h)                   # (N, hidden)

            # Skip/residual con proyección de entrada
            xh = xh + h

        delta_norm = self.head(h)                    # (N, out_ch)
        return delta_norm, h

    # Azúcar sintáctico para usarla como “un paso”
    def step(self, x, edge_index, edge_attr, h_prev=None):
        return self.forward(x, edge_index, edge_attr, h_prev)


## Loss functions

In [ ]:
def smooth_edge_penalty(pred, target, edge_index, lam=1e-3,
                        bc_mask=None, solid_id=None):
    """Match the gradient of the edge pred vs target, but ignores
    the edges that have nodes in the BC or belong to different solids."""
    src, dst = edge_index
    diff = (pred[src] - pred[dst]) - (target[src] - target[dst])  # [E, C]

    if bc_mask is not None:
        free_edge = ((bc_mask[src] == 0) & (bc_mask[dst] == 0)).unsqueeze(-1)  # [E,1]
        diff = diff * free_edge

    if solid_id is not None:
        same_solid = (solid_id[src] == solid_id[dst]).unsqueeze(-1)  # [E,1]
        diff = diff * same_solid

    return lam * diff.pow(2).mean()

def masked_mse(pred, target, mask_free):
    # mask_free: True en nodos libres
    if mask_free is None: return F.mse_loss(pred, target)
    pred_f, tgt_f = pred[mask_free], target[mask_free]
    return F.mse_loss(pred_f, tgt_f)

def loss_bc_zero_disp(pred_norm, dynamic_features, bc_mask,
                      y_scaler, lam_bc: float = 1e-3):
    """
    Penaliza desplazamiento != 0 EN ESPACIO FÍSICO en nodos fijos (bc_mask=True).
    pred_norm: y_hat normalizado
    y_scaler: (mean, std) usados para normalizar y. Si None, asumimos ya absoluto.
    """
    # TODO: Include custom weights for each features(some could be noisier)
    if lam_bc <= 0 or bc_mask is None or bc_mask.sum() == 0:
        return pred_norm.new_tensor(0.0)
    if y_scaler is None:
        pred_phys = pred_norm
    else:
        ym, ys = y_scaler
        pred_phys = pred_norm * ys.to(pred_norm) + ym.to(pred_norm)
    dynamic_features = pred_phys[:, dynamic_features]
    return lam_bc * (dynamic_features[bc_mask] ** 2).mean()

def kinematic_consistency(y_next_phys, x_prev_phys,
                          disp_dims=(0,1,2), vel_dims=(3,4,5),
                          dt=1.0, lam=1e-3):
    """
    Fuerza v_{t+1} ≈ (Δx_{t+1} - Δx_{t}) / dt en nodos libres.
    y_next_phys: estado dinámico en t+1 (físico) [N, Ddyn]
    x_prev_phys: estado completo en t   (físico) [N, Din] -> usaremos sus dinámicos
    """
    if len(vel_dims) == 0 or max(vel_dims) >= y_next_phys.size(1):
        return y_next_phys.new_tensor(0.0)

    disp_next = y_next_phys[:, disp_dims]
    vel_next  = y_next_phys[:, vel_dims]
    disp_prev = x_prev_phys[:, :][:, disp_dims]  # si tus dinámicos están al inicio del bloque de x

    vel_from_disp = (disp_next - disp_prev) / dt
    return lam * F.smooth_l1_loss(vel_next, vel_from_disp)


def masked_smooth_l1(pred, target, mask=None, beta=1.0):
    """
    SmoothL1 (Huber) con máscara booleana opcional.
    """
    if mask is None:
        return F.smooth_l1_loss(pred, target, beta=beta)
    return F.smooth_l1_loss(pred[mask], target[mask], beta=beta)


def masked_smooth_l1_weighted(
    pred,                 # [N, C] o [B, N, C]
    target,               # misma forma que pred
    *,
    valid_mask=None,      # [N, C] o [B, N, C] bool (opcional)
    feat_weights=None,    # [C] (opcional)
    node_weights=None,    # [N] o [B, N] (opcional)
    beta=1.0,
    eps=1e-12,
):
    """
    Huber per-element -> *pesos* -> promedio ponderado.
    """
    loss = F.smooth_l1_loss(pred, target, beta=beta, reduction='none')  # misma forma que pred
    w = torch.ones_like(loss)

    # máscara de validez
    if valid_mask is not None:
        w = w * valid_mask.to(w.dtype)

    # pesos por feature
    if feat_weights is not None:
        # feat_weights: [C]
        w = w * feat_weights.view(([1] * (loss.dim()-1)) + [-1])  # broadcast al último dim

    # pesos por nodo
    if node_weights is not None:
        # node_weights: [N] o [B, N]
        w = w * node_weights.view(list(loss.shape[:-1]) + [1])    # broadcast a todas las features

    # promedio ponderado robusto
    weighted = loss * w
    denom = w.sum().clamp_min(eps)
    return weighted.sum() / denom

def edge_geometric_loss(
    y_hat_phys: torch.Tensor,        # [N,3] Δ̂_{t+1} en físico
    y_true_phys: torch.Tensor,       # [N,3] Δ_{t+1} GT en físico
    pos0: torch.Tensor,              # [N,3] posiciones iniciales
    edge_index: torch.Tensor,        # [2,E]
    bc_mask: torch.Tensor = None,    # [N] bool/0-1 (opcional) -> excluye aristas que toquen BC
    solid_id: torch.Tensor = None,   # [N] long (opcional) -> id de sólido por nodo
    ignore_intersolid: bool = True,  # ignora aristas entre sólidos distintos
    lam_vec: float = 0.0,            # peso del término vectorial (pred-true)
    lam_norm: float = 1.0,           # peso del término de norma (longitud)
    lam_ang: float = 0.5,            # peso del término angular (1-cos)
    relative: bool = True,           # normalizar por d0 (strain-like)
    beta: float = 0.01,              # beta para SmoothL1
    eps: float = 1e-8
):
    device = y_hat_phys.device
    src, dst = edge_index.to(device)

    P_pred = pos0.to(device) + y_hat_phys # predicted position
    P_true = pos0.to(device) + y_true_phys # real position

    e_pred = P_pred[src] - P_pred[dst]                       # [E,3]
    e_true = P_true[src] - P_true[dst]                       # [E,3]

    d_pred = e_pred.norm(dim=-1, keepdim=True).clamp_min(eps)  # [E,1]
    d_true = e_true.norm(dim=-1, keepdim=True).clamp_min(eps)  # [E,1]
    d0     = (pos0[src] - pos0[dst]).norm(dim=-1, keepdim=True).clamp_min(eps)  # [E,1]

    # ---- máscara de aristas válidas ----
    mask_e = torch.ones(e_pred.size(0), 1, dtype=torch.bool, device=device)

    if bc_mask is not None:
        m = (bc_mask.to(device) != 0).view(-1)               # [N] bool
        mask_e &= (~m[src] & ~m[dst]).unsqueeze(-1)          # ambas puntas libres

    if solid_id is not None and ignore_intersolid:
        sid = solid_id.to(device).view(-1).long()            # [N] ids de sólido
        same = (sid[src] == sid[dst]).unsqueeze(-1)          # [E,1]
        mask_e &= same                                       # solo intra-sólido

    # si no queda ninguna arista válida, devolver 0 (evita .mean() sobre vacío)
    if not mask_e.any():
        return y_hat_phys.new_zeros(())

    losses = []

    if lam_vec != 0.0:
        vec_diff = (e_pred - e_true) / (d0 if relative else 1.0)     # [E,3]
        z = torch.zeros_like(vec_diff)
        l_vec = F.smooth_l1_loss(vec_diff[mask_e.expand_as(vec_diff)],
                                 z[mask_e.expand_as(vec_diff)],
                                 beta=beta, reduction='mean')
        losses.append(lam_vec * l_vec)

    if lam_norm != 0.0:
        dn = (d_pred - d_true) / (d0 if relative else 1.0)           # [E,1]
        l_norm = F.smooth_l1_loss(dn[mask_e], torch.zeros_like(dn[mask_e]),
                                  beta=beta, reduction='mean')
        losses.append(lam_norm * l_norm)

    if lam_ang != 0.0:
        cos = (e_pred * e_true).sum(-1, keepdim=True) / (d_pred * d_true)  # [E,1]
        ang = 1.0 - cos.clamp(-1.0, 1.0)                                   # [E,1]
        l_ang = ang[mask_e].mean()
        losses.append(lam_ang * l_ang)

    return sum(losses) if losses else y_hat_phys.new_zeros(())

# Prob. de teacher forcing (decae 1.0 -> 0.2 en 50 épocas, ajusta a gusto)
def p_teacher(epoch, p0=1.0, pmin=0.2, T=30):
  '''
  Con qué frecuencia usamos el ground truth en lugar de la prediccón.
  Si el rollout de validación explota,sube p-teacher(mayor pmin o mayor T o reduce K)
  Si el train baja muy lento, quizas p-teacher muy bajo
  '''
  return max(pmin, p0 - (p0 - pmin) * epoch / max(T, 1))



## Data augmentation

In [ ]:
def create_geom_node_features(x_t_phys, dyn_idx_x_t, pos0, pos_scaler, device, pos_norm_mode, pos_scale):
  pos0 = pos0.to(x_t_phys).float()
  disp_t = x_t_phys[:, dyn_idx_x_t][:, :3]  # [dx,dy,dz] en físico
  pos_t = pos0 + disp_t

  if pos_norm_mode == "dataset":
      pm_d, ps_d = pos_scaler[0].to(device), pos_scaler[1].to(device)
      pos_t_norm = (pos_t - pm_d) / ps_d
  elif pos_norm_mode == "center_graph":
      mu = pos0.mean(0, keepdim=True)
      pos_t_norm = pos_t - mu
  elif pos_norm_mode == "graph_std":
      mu = pos0.mean(0, keepdim=True); sig = pos0.std(0, keepdim=True).clamp_min(1e-6)
      pos_t_norm = (pos_t - mu) / sig
  else:
      raise ValueError(f"POS_NORM_MODE desconocido: {pos_norm_mode}")
  pos_t_norm = pos_scale * pos_t_norm
  return pos_t_norm

@torch.no_grad()
def create_geom_edge_features(
    x_for_edges_phys: torch.Tensor,                 # (N, Cx) en físico
    dyn_idx_x_t: Sequence[int],                     # índices de x donde están [dx,dy,dz,...]
    pos0: torch.Tensor,                             # (N, 3) en físico
    edge_index: torch.Tensor,                       # (2, E) long
    edge_geom_scaler: Optional[Tuple[torch.Tensor, torch.Tensor]] = None, # (em, es) de (E,4)
    edge_attr: Optional[torch.Tensor] = None,       # (E, Ce) ya normalizado si aplica
    edge_dyn_scale: float = 1.0,
    eps: float = 1e-8,
) -> Optional[torch.Tensor]:
    """
    Devuelve edge_in:
      - si use_edge_dyn: concatena [rel_x, rel_y, rel_z, dist] normalizados + edge_attr (si existe).
      - si no: devuelve edge_attr (puede ser None).
    Asume tensores 2D (N,C) para nodos y (E,C) para aristas (PyG estándar).
    """
    if isinstance(dyn_idx_x_t, torch.Tensor):
      dyn_idx_x_t = dyn_idx_x_t.detach().cpu().tolist()
    else:
      dyn_idx_x_t = list(dyn_idx_x_t)


    # --- checks básicos ---
    if len(dyn_idx_x_t) < 3:
        raise ValueError("dyn_idx_x_t debe contener al menos [dx,dy,dz].")
    if pos0.shape[-1] != 3:
        raise AssertionError("pos0 debe tener última dimensión = 3.")
    if edge_index.ndim != 2 or edge_index.size(0) != 2:
        raise AssertionError("edge_index debe ser de forma (2, E).")

    # --- devices & dtypes ---
    x_for_edges_phys = x_for_edges_phys.float()
    pos0 = pos0.to(x_for_edges_phys).float()
    edge_index = edge_index.to(x_for_edges_phys.device)
    if edge_index.dtype != torch.long:
        edge_index = edge_index.long()

    # --- construir pos_edges = pos0 + [dx,dy,dz] ---
    disp_edges = x_for_edges_phys[..., list(dyn_idx_x_t)][..., :3]  # (N, 3)
    pos_edges  = pos0 + disp_edges                                  # (N, 3)

    # --- geometría de aristas: rel y distancia ---
    s, d = edge_index[0], edge_index[1]        # (E,), (E,)
    rel  = pos_edges[s] - pos_edges[d]         # (E, 3)
    dist = rel.norm(dim=-1, keepdim=True)      # (E, 1)
    edge_dyn = torch.cat([rel, dist], dim=-1)  # (E, 4)

    # --- normalización geométrica (dataset) ---
    if edge_geom_scaler is not None:
        em, es = edge_geom_scaler
        em = em.to(edge_dyn)
        es = es.to(edge_dyn).clamp_min(eps)
        edge_dyn = (edge_dyn - em) / es

    # --- re-escala opcional ---
    if edge_dyn_scale != 1.0:
        edge_dyn = edge_dyn * float(edge_dyn_scale)

    # --- concatenación con edge_attr (si existe) ---
    if edge_attr is not None:
        edge_attr = edge_attr.to(edge_dyn)  # device & dtype match
        edge_in = torch.cat([edge_attr, edge_dyn], dim=-1)
    else:
        edge_in = edge_dyn

    return edge_in

@torch.no_grad()
def add_world_edges_and_features(
    pos_edges: torch.Tensor,            # (N,3) posiciones físicas (pred-only track)
    edge_index_mesh: torch.Tensor,      # (2,E_mesh)
    edge_attr_static: torch.Tensor | None,
    r_world: float,
    edge_geom_scaler: tuple[torch.Tensor, torch.Tensor] | None = None,
):
    # 1) construir world edges por radio
    try:
        from torch_cluster import radius_graph
        ei_world = radius_graph(pos_edges, r=r_world, loop=False)  # (2, Ew)
    except Exception:
        # Fallback O(N^2) si no está torch_cluster (para N pequeño)
        N = pos_edges.size(0)
        rel = pos_edges.unsqueeze(1) - pos_edges.unsqueeze(0)  # (N,N,3)
        dist = rel.norm(dim=-1)
        mask = (dist <= r_world) & (~torch.eye(N, dtype=torch.bool, device=pos_edges.device))
        s, d = mask.nonzero(as_tuple=True)
        ei_world = torch.stack([s, d], dim=0)

    # 2) elimina duplicados con la malla (no dirigido)
    def undirected_pairs(ei):
        a = torch.minimum(ei[0], ei[1]); b = torch.maximum(ei[0], ei[1])
        return torch.stack([a, b], dim=1)
    if edge_index_mesh is not None and edge_index_mesh.numel() > 0:
        mesh_ud  = undirected_pairs(edge_index_mesh)
        world_ud = undirected_pairs(ei_world)
        key_mesh  = mesh_ud[:,0] * pos_edges.size(0) + mesh_ud[:,1]
        key_world = world_ud[:,0] * pos_edges.size(0) + world_ud[:,1]
        keep = ~torch.isin(key_world, key_mesh)
        ei_world = ei_world[:, keep]

    # 3) combina conectividad
    ei = edge_index_mesh if ei_world.numel() == 0 else torch.cat([edge_index_mesh, ei_world], dim=1)

    # 4) atributos geométricos [rel, dist] sobre EI combinado
    s, d = ei[0], ei[1]
    rel  = pos_edges[s] - pos_edges[d]                 # (E,3)
    dist = rel.norm(dim=-1, keepdim=True)              # (E,1)
    edge_dyn = torch.cat([rel, dist], dim=-1)          # (E,4)

    # 5) normaliza con tu scaler (si lo tienes)
    if edge_geom_scaler is not None:
        em, es = edge_geom_scaler
        edge_dyn = (edge_dyn - em.to(edge_dyn)) / es.to(edge_dyn).clamp_min(1e-8)

    # 6) concat con estáticos si existen (rellena mundo con 0 si edge_attr_static sólo cubre malla)
    if edge_attr_static is not None:
        Ce = edge_attr_static.size(-1)
        E_mesh = edge_index_mesh.size(1)
        E_total = ei.size(1)
        if E_total == E_mesh:
            edge_in = torch.cat([edge_attr_static.to(edge_dyn), edge_dyn], dim=-1)
        else:
            zeros_world = torch.zeros(E_total - E_mesh, Ce, device=edge_dyn.device, dtype=edge_dyn.dtype)
            edge_attr_all = torch.cat([edge_attr_static.to(edge_dyn), zeros_world], dim=0)
            edge_in = torch.cat([edge_attr_all, edge_dyn], dim=-1)
    else:
        edge_in = edge_dyn

    # (opcional) podrías añadir un flag de tipo si tu modelo lo usa:
    # edge_type = torch.cat([torch.zeros(E_mesh,1,device=ei.device), torch.ones(E_total-E_mesh,1,device=ei.device)], dim=0)
    # edge_in = torch.cat([edge_in, edge_type], dim=-1)

    if ei.dtype != torch.long: ei = ei.long()
    return ei, edge_in


@torch.no_grad()
def make_dynamic_edge_inputs(
    pos0: torch.Tensor,                          # (N,3)
    x_for_edges_phys: torch.Tensor,              # (N, Cx) track pred-only
    dyn_idx_x_t: Sequence[int],                  # índices dinámicos; dx,dy,dz en los 3 primeros
    edge_index_mesh: torch.Tensor,               # (2, E_mesh) long
    edge_attr_static: Optional[torch.Tensor],    # (E_mesh, Ce) o None (ya normalizado si aplica)
    use_edge_dyn: bool,                          # master switch
    use_world_edges: bool,                       # nuevo enfoque
    r_world: float = 0.0,                        # radio de proximidad (si world)
    edge_geom_scaler: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    edge_dyn_scale: float = 1.0,
):
    """
    Devuelve (edge_index_for_model, edge_in) según flags:
      - if not use_edge_dyn: (edge_index_mesh, edge_attr_static)
      - elif use_world_edges: world-edges + geom features -> concat con estáticos
      - else: geom features sobre edge_index_mesh -> concat con estáticos
    """
    # Sin dinámicas de arista: usar sólo estáticos
    if not use_edge_dyn:
        ei = edge_index_mesh
        edge_in = edge_attr_static
        if ei.dtype != torch.long: ei = ei.long()
        return ei, edge_in

    # Posiciones para edges = pos0 + [dx,dy,dz] de la pista pred-only
    disp = x_for_edges_phys[..., list(dyn_idx_x_t)][..., :3]  # (N,3)
    pos_edges = (pos0.to(x_for_edges_phys) + disp).float()    # (N,3)

    if use_world_edges:
        # ---- enfoque nuevo: world edges ----
        ei_for_model, edge_in = add_world_edges_and_features(
            pos_edges=pos_edges,
            edge_index_mesh=edge_index_mesh,
            edge_attr_static=edge_attr_static,
            r_world=r_world,
            edge_geom_scaler=edge_geom_scaler,
        )
        # (add_world_edges_and_features ya calcula [rel,dist], normaliza y concatena)
        return ei_for_model, edge_in

    # ---- enfoque antiguo: sólo malla + geom dyn ----
    edge_in = create_geom_edge_features(
        x_for_edges_phys=x_for_edges_phys,
        dyn_idx_x_t=dyn_idx_x_t,
        pos0=pos0,
        edge_index=edge_index_mesh,
        edge_geom_scaler=edge_geom_scaler,
        edge_attr=edge_attr_static,
        edge_dyn_scale=edge_dyn_scale,
    )
    ei = edge_index_mesh if edge_index_mesh.dtype == torch.long else edge_index_mesh.long()
    return ei, edge_in

def schedule_sigma(epoch: int,
                   sigma_start: float = 0.0,
                   sigma_max: float = 1.0,
                   warmup_epochs: int = 10) -> float:
    """
    σ lineal desde sigma_start -> sigma_max durante warmup_epochs.
    Devuelve un factor *adimensional*; lo convertiremos a unidades físicas abajo.
    """
    if warmup_epochs <= 0:
        return sigma_max
    p = min(max(epoch / float(warmup_epochs), 0.0), 1.0)
    return sigma_start + p * (sigma_max - sigma_start)

@torch.no_grad()
def add_noise_to_state(
    x_t_phys: torch.Tensor,           # (N, Din) físico
    dyn_idx_x: Sequence[int],         # índices dinámicos dentro de x
    bc_mask: torch.Tensor | None,     # [N] bool (True=fijo)
    *,
    sigma_disp_phys: float = 0.0,     # σ para [dx,dy,dz] en unidades físicas
    sigma_vel_phys: float = 0.0,      # σ para [vx,vy,vz] en unidades físicas
    has_velocity: bool = True,        # si existen velocidades en dyn_idx_x
    vel_offset_from_disp: bool = True,# hace el ruido de v coherente con Δx
    dt: float = 1.0,                  # para coherencia v ≈ Δx/dt
) -> torch.Tensor:
    """
    Devuelve x_t_phys + ruido (sólo en nodos libres y features dinámicas).
    - Aplica N(0, σ^2) en desplazamientos.
    - Si has_velocity y vel_offset_from_disp: usa el mismo offset/ dt para v.
    """
    x_noisy = x_t_phys.clone()
    device  = x_t_phys.device

    if isinstance(dyn_idx_x, torch.Tensor):
        dyn_idx_x = dyn_idx_x.detach().cpu().tolist()
    dyn_idx_x = list(dyn_idx_x)

    # Mapeo simple: asumimos [dx,dy,dz] están al principio del bloque dinámico.
    # Si tus índices son distintos, ajusta aquí.
    disp_cols_global = [dyn_idx_x[0], dyn_idx_x[1], dyn_idx_x[2]]
    vel_cols_global  = [dyn_idx_x[3], dyn_idx_x[4], dyn_idx_x[5]] if (has_velocity and len(dyn_idx_x) >= 6) else []

    N = x_t_phys.size(0)
    free_mask = torch.ones(N, dtype=torch.bool, device=device)
    if bc_mask is not None:
        free_mask = ~bc_mask.to(device)

    # --- Ruido en desplazamientos ---
    if sigma_disp_phys > 0.0:
        eps_disp = torch.randn((N, 3), device=device) * float(sigma_disp_phys)
        eps_disp[~free_mask] = 0.0
        x_noisy[:, disp_cols_global] = x_noisy[:, disp_cols_global] + eps_disp

        # --- Ruido coherente en velocidades (opcional) ---
        if vel_cols_global:
            if vel_offset_from_disp and sigma_vel_phys == 0.0:
                # mism o desplazamiento dividido por dt
                eps_vel = eps_disp / float(dt)
            else:
                eps_vel = torch.randn((N, 3), device=device) * float(sigma_vel_phys)
            eps_vel[~free_mask] = 0.0
            x_noisy[:, vel_cols_global] = x_noisy[:, vel_cols_global] + eps_vel

    else:
        # Sólo velocidades si se pide
        if vel_cols_global and sigma_vel_phys > 0.0:
            eps_vel = torch.randn((N, 3), device=device) * float(sigma_vel_phys)
            eps_vel[~free_mask] = 0.0
            x_noisy[:, vel_cols_global] = x_noisy[:, vel_cols_global] + eps_vel

    return x_noisy


Train

## train_epoch

In [ ]:
def train_epoch_k(
    model,
    train_static: dict,
    x_scaler,                # (xm, xs)  para normalizar x_t
    delta_scaler,            # (dm, ds)  para desnormalizar Δ
    dyn_idx_x,               # índices dinámicos en x (usa X_DYNAMIC_INDEX)
    edge_scaler=None,
    edge_geom_scaler=None,
    pos_scaler=None,
    device='cuda',
    epoch=1,
    lam_smooth=1e-3,
    lam_bc=1e-2,
    k_min=1,
    k_max=8,
    scaler=None,
    opt=None,
    max_grad_norm=1.0,
    attributes_weights = None,
    scheduler = None
):
    """
    Entrena con ventanas aleatorias de longitud K (teacher forcing programado).
    Trabaja SIEMPRE en espacio físico.
    """
    assert opt is not None, "Falta optimizer"
    model.train()

    xm, xs = x_scaler
    dm, ds = delta_scaler
    pm, ps = pos_scaler
    em, es = edge_scaler
    xm_d, xs_d = xm.to(device), xs.to(device)
    dm_d, ds_d = dm.to(device), ds.to(device)
    pm_d, ps_d = pm.to(device), ps.to(device)
    em_d, es_d = em.to(device), es.to(device)

    dyn_idx_x_t = torch.as_tensor(dyn_idx_x, device=device)
    disp_dims = torch.as_tensor([0, 1, 2], device=device)  # clamp y BC sólo en desplazamiento


    # Asumimos que los 3 primeros dims dinámicos de Δ son [dx,dy,dz]
    # y los 3 siguientes (si existen) son [vx,vy,vz].
    sigma_factor = schedule_sigma(epoch, 0.0, 1.0, NOISE_WARMUP_EPOCHS) if USE_INPUT_NOISE else 0.0

    # std físicos por feature de Δ:
    ds_cpu = ds.detach().cpu().view(-1).numpy()
    # indices relativos dentro de dyn block
    disp_rel = [0,1,2]
    vel_rel  = [3,4,5] if (len(dyn_idx_x) >= 6) else []

    # coge std de Δ (físico) en esas columnas dinámicas
    def std_of(rel_cols):
        if not rel_cols: return 0.0
        # mapea rel->global en y/delta: como tu delta predice exactamente las mismas dims que dinámicas
        return float(torch.tensor([ds[0, idx] for idx in rel_cols]).mean().item())

    ds_disp = std_of(disp_rel)  # escala de Δx por paso
    ds_vel  = std_of(vel_rel)   # escala de Δv por paso (si hay)

    sigma_disp_phys = None
    sigma_vel_phys  = None
    if k_max < 10:
      sigma_disp_phys = 0.15 * ds_disp * sigma_factor
      sigma_vel_phys  = 0.10  * ds_vel  * sigma_factor
    elif k_max < 30:
      sigma_disp_phys = 0.20 * ds_disp * sigma_factor
      sigma_vel_phys  = 0.12  * ds_vel  * sigma_factor
    elif k_max < 90:
      sigma_disp_phys = 0.25 * ds_disp * sigma_factor
      sigma_vel_phys  = 0.15  * ds_vel  * sigma_factor
    else:
      sigma_disp_phys = 0.25 * ds_disp * sigma_factor
      sigma_vel_phys  = 0.15  * ds_vel  * sigma_factor



    feat_w = torch.tensor(attributes_weights, device=device)

    total_loss, total_nodes = 0.0, 0
    sim_ids = list(train_static.keys())
    random.shuffle(sim_ids)
    pbar = tqdm(sim_ids, desc=f"Train (epoch {epoch})", leave=False)

    for sid in pbar:
        info = train_static[sid]
        edge_index = info['edge_index'].to(device)
        edge_attr  = info.get('edge_attr', None)
        if edge_attr is not None and edge_scaler is not None:
            edge_attr = (edge_attr - em) / es
        edge_attr  = edge_attr.to(device) if edge_attr is not None else None

        bc_mask    = info.get('bc_mask', None)
        if bc_mask is not None: bc_mask = bc_mask.to(device).bool()
        mask_free = None if bc_mask is None else ~bc_mask
        rigid_mask = info.get('rigid_mask', None)
        if rigid_mask is not None: rigid_mask = rigid_mask.to(device)

        y_real_phys = info['y_real'].to(device)  # (T-1, N, Ddyn) en físico
        x_t_phys    = info['x0'].to(device).clone()  # (N, Din) en físico
        pos0 = info['pos0'].to(device).float()
        Tm1, N, Ddyn = y_real_phys.shape

        if Tm1 < 1:
            pbar.set_postfix(skip="Tm1<1")
            continue

        # Selecciona ventana aleatoria [t0, t0+K)
        t0 = int(torch.randint(0, max(1, Tm1 - k_max + 1), (1,)).item())
        K  = int(torch.randint(k_min, min(k_max, Tm1 - t0) + 1, (1,)).item())

        if t0 > 0:
          # Estado en t0 para el bloque dinámico = y_real[t0-1]
          x_t_phys[:, dyn_idx_x_t] = y_real_phys[t0 - 1]


        # K  = min(k_max, Tm1 - t0)

        # x_for_edges_phys = x_t_phys.clone()
        x_for_edges_phys = x_t_phys

        opt.zero_grad(set_to_none=True)
        # with torch.cuda.amp.autocast(enabled=(scaler is not None)):

        loss_sum_for_log = 0.0
        for k in range(K):
            with torch.no_grad():
              if USE_INPUT_NOISE and (sigma_disp_phys > 0.0 or sigma_vel_phys > 0.0):
                  x_in_phys = add_noise_to_state(
                      x_t_phys,
                      dyn_idx_x=dyn_idx_x_t,
                      bc_mask=bc_mask,
                      sigma_disp_phys=sigma_disp_phys,
                      sigma_vel_phys=sigma_vel_phys,
                      has_velocity=(len(dyn_idx_x) >= 6),
                      vel_offset_from_disp=VEL_FROM_DISP,
                      dt=DT_TRAIN,
                  )
              else:
                  x_in_phys = x_t_phys

              # Normaliza entrada del paso (x_t)
              x_in_norm = (x_in_phys - xm_d) / xs_d
              x_in = None
              if AUGMENT_GEOM_NODE_FEATURES:
                x_in = torch.cat([x_in_norm, create_geom_node_features(x_t_phys, dyn_idx_x, pos0, pos_scaler, device, POS_NORM_MODE, POS_SCALE)], dim=1)
              else:
                x_in = x_in_norm
              ei_for_model, edge_in = make_dynamic_edge_inputs(
                  pos0=pos0,
                  x_for_edges_phys=x_for_edges_phys,     # track pred-only
                  dyn_idx_x_t=dyn_idx_x_t,
                  edge_index_mesh=edge_index,
                  edge_attr_static=edge_attr,            # estáticos (normalizados si procede)
                  use_edge_dyn=USE_EDGE_DYN,
                  use_world_edges=USE_WORLD_EDGES,       # << tu flag nuevo
                  r_world=R_WORLD,                       # hiperparámetro
                  edge_geom_scaler=edge_geom_scaler,
                  edge_dyn_scale=EDGE_DYN_SCALE,
              )

            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
              # Predice Δ en normalizado y pásalo a físico con delta_scaler
              if k == 0: h = None
              delta_norm, h = model(x_in, ei_for_model, edge_in, h)         # (N, Ddyn)
              h = h.detach() # para que no explote pero igual peores resultados
              # delta_norm = model(x_in, edge_index, edge_in)         # (N, Ddyn)
              delta_phys = delta_norm * ds_d + dm_d                         # (N, Ddyn)

              # Estado siguiente predicho (RESIDUAL)
              y_hat_phys = x_t_phys[:, dyn_idx_x_t] + delta_phys


              loss  = smooth_edge_penalty(y_hat_phys, y_real_phys[t0+k], edge_index, lam_smooth, bc_mask, rigid_mask)

              expanded_mask_free = mask_free.unsqueeze(-1).expand_as(y_hat_phys) if mask_free is not None else None
              loss += masked_smooth_l1_weighted(y_hat_phys, y_real_phys[t0+k], feat_weights= feat_w, beta = SMOOTHL1_BETA, valid_mask=expanded_mask_free)

              # BC: usa SmoothL1 contra 0 sólo en desplazamientos
              if bc_mask is not None and bc_mask.any():
                  zero = torch.zeros_like(y_hat_phys[bc_mask][:, disp_dims])
                  loss += lam_bc * F.smooth_l1_loss(y_hat_phys[bc_mask][:, disp_dims], zero, beta=SMOOTHL1_BETA)

              # -----------------------------------------------------

              if LAM_GEO > 0:
                  loss += LAM_GEO * edge_geometric_loss(y_hat_phys[:,:3], y_real_phys[t0+k][:,:3], pos0=info['pos0'].to(device), edge_index=edge_index, bc_mask=bc_mask, solid_id=rigid_mask, ignore_intersolid=True, lam_vec=LAM_VEC, lam_norm=LAM_NORM, lam_ang=LAM_ANG, relative=True, beta=SMOOTHL1_BETA)

              if LAM_KIN > 0:
                if len(VEL_DIMS) > 0 and (max(VEL_DIMS) < y_hat_phys.size(1)):
                    disp_prev = x_t_phys[:, dyn_idx_x_t][:, DISP_DIMS]
                    disp_next = y_hat_phys[:, DISP_DIMS]
                    vel_next  = y_hat_phys[:, VEL_DIMS]
                    vel_from_disp = (disp_next - disp_prev) / DT
                    loss += LAM_KIN * F.smooth_l1_loss(vel_next, vel_from_disp, beta=SMOOTHL1_BETA)

              # === BACKWARD POR PASO: evita retener K grafos ===
              if scaler is not None:
                  scaler.scale(loss / float(K)).backward()
              else:
                  (loss / float(K)).backward()

              # Solo para logging/promedios (sin retener grafo)
              loss_sum_for_log += float(loss.detach())


              # Scheduled sampling para el siguiente estado
              use_gt = (torch.rand(1, device=device).item() < p_teacher(epoch, P_TEACHER_P0, P_TEACHER_PMIN, P_TEACHER_WARMUP))
              x_next_dyn = y_real_phys[t0 + k] if use_gt else y_hat_phys.detach()

              # x_t_phys = x_t_phys.clone()
              x_t_phys = x_t_phys
              x_t_phys[:, dyn_idx_x_t] = x_next_dyn
              # x_for_edges_phys = x_t_phys.clone()
              x_for_edges_phys = x_t_phys.clone()
              x_for_edges_phys[:, dyn_idx_x_t] = (x_next_dyn if use_gt else y_hat_phys).detach()

              del delta_norm, loss, edge_in, ei_for_model

        # loss_final = loss_accum / float(K)

        # Backward
        if scaler is not None:
            # scaler.scale(loss_final).backward()
            if max_grad_norm is not None:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(opt); scaler.update()
            if scheduler is not None:
              scheduler.step()
        else:
            # loss_final.backward()
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
            if scheduler is not None:
              scheduler.step()

        # total_loss += loss_final.item() * N
        loss_final_val = loss_sum_for_log / float(K)
        total_loss += loss_final_val * N
        total_nodes += N

        avg = total_loss / max(total_nodes, 1)
        pbar.set_postfix(K=K, t0=t0, loss=f"{avg:.4f}")

    return avg


Eval

In [ ]:
@torch.no_grad()
def compute_metrics(
    y_pred: torch.Tensor,
    y_real: torch.Tensor,
    only_displacement: bool = False,
    disp_dims: Sequence[int] = (0, 1, 2),
) -> Dict[str, float]:
    """
    - Soporta formas (T,N,D) o (N,D). Siempre computa normas en la última dim.
    - ADE: media de ||error||2 a lo largo de todo (y tiempo si hay).
    - FDE (si hay T): media de ||error_T||2 en el último paso temporal.
      Si no hay T (N,D), se devuelve FDE=ADE.
    """
    assert y_pred.shape == y_real.shape, "Shapes distintas entre y_pred e y_real"
    if only_displacement:
        sel = list(disp_dims)
        y_pred = y_pred[..., sel]
        y_real = y_real[..., sel]

    diff = y_pred - y_real

    # MAE y RMSE globales (sobre todos los elementos)
    mae  = diff.abs().mean().item()
    rmse = torch.sqrt((diff ** 2).mean()).item()

    # L2 por muestra (y por tiempo si aplica)
    l2 = torch.norm(diff, dim=-1)  # -> (T,N) o (N,)

    ade = l2.mean().item()

    if diff.ndim == 3:  # (T,N,D)
        final_l2 = torch.norm(y_pred[-1] - y_real[-1], dim=-1)  # (N,)
        fde = final_l2.mean().item()
    else:               # (N,D) -> no hay tiempo: toma FDE=ADE
        fde = ade

    return {"MAE": mae, "RMSE": rmse, "ADE": ade, "FDE": fde}

## eval_epoch

In [ ]:
@torch.no_grad()
def eval_epoch_k(
    model, val_static, x_scaler, delta_scaler, dyn_idx_x,
    edge_scaler=None, edge_geom_scaler=None,
    pos_scaler=None, device='cuda',
    lam_smooth=1e-3, lam_bc=1e-2, K_eval=8, attributes_weights=None,
    teacher_forcing=True, sample_t0=False,
    use_tqdm=True,
    return_metrics: bool = False,          # <-- NUEVO: para devolver métricas agregadas
    only_disp_in_logs: bool = True,        # <-- logs centrados en desplazamientos
):
    model.eval()

    xm, xs = x_scaler
    dm, ds = delta_scaler
    xm_d, xs_d = xm.to(device), xs.to(device)
    dm_d, ds_d = dm.to(device), ds.to(device)

    dyn_idx_x_t = torch.as_tensor(dyn_idx_x, device=device)
    disp_dims = torch.as_tensor([0, 1, 2], device=device)
    feat_w = torch.tensor(attributes_weights, device=device) if attributes_weights is not None else None

    sim_ids = list(val_static.keys())

    total_loss, total_nodes = 0.0, 0

    # ---- NUEVO: acumuladores globales de métricas ----
    ADE_all_sum = 0.0; FDE_all_sum = 0.0
    ADE_disp_sum = 0.0; FDE_disp_sum = 0.0
    n_sims_acc = 0

    # para diagnóstico de “estallido”
    per_t_ade_disp_all = []    # lista de tensores (K,) por simulación
    bc_violation_max_all = []  # max ||disp|| en BC por sim (en cualquier t)
    nan_infs = 0               # contador de sims con NaN/Inf

    pbar = tqdm(sim_ids, desc="Valid", leave=False) if use_tqdm else sim_ids

    for sid in pbar:
        info = val_static[sid]
        edge_index = info['edge_index'].to(device)
        edge_attr  = info.get('edge_attr', None)
        if edge_attr is not None and edge_scaler is not None:
            edge_attr = (edge_attr - edge_scaler[0]) / edge_scaler[1]
        edge_attr  = edge_attr.to(device) if edge_attr is not None else None

        bc_mask    = info.get('bc_mask', None)
        if bc_mask is not None: bc_mask = bc_mask.to(device).bool()
        mask_free = None if bc_mask is None else ~bc_mask
        rigid_mask = info.get('rigid_mask', None)
        if rigid_mask is not None: rigid_mask = rigid_mask.to(device)

        y_real_phys = info['y_real'].to(device)
        x_t_phys    = info['x0'].to(device).clone()
        pos0 = info['pos0'].to(device).float()

        Tm1, N, Ddyn = y_real_phys.shape
        if Tm1 < 1:
            if use_tqdm: pbar.set_postfix(skip="Tm1<1")
            continue

        # t0 y K (autoregresivo)
        if sample_t0 and Tm1 > K_eval:
            t0 = int(torch.randint(0, Tm1 - K_eval + 1, (1,)).item())
        else:
            t0 = 0
        K = min(K_eval, Tm1 - t0)

        if t0 > 0:
            x_t_phys[:, dyn_idx_x_t] = y_real_phys[t0 - 1]

        x_for_edges_phys = x_t_phys.clone()

        loss_accum = 0.0

        # ---- NUEVO: almacenar secuencias para métricas ADE/FDE ----
        pred_steps = []
        real_steps = []

        exploded_naninf = False
        with torch.no_grad():
          for k in range(K):
              x_in_norm = (x_t_phys - xm_d) / xs_d
              x_in = torch.cat([x_in_norm, create_geom_node_features(x_t_phys, dyn_idx_x, pos0, pos_scaler, device, POS_NORM_MODE, POS_SCALE)], dim=1) \
                      if AUGMENT_GEOM_NODE_FEATURES else x_in_norm

              ei_for_model, edge_in = make_dynamic_edge_inputs(
                  pos0=pos0,
                  x_for_edges_phys=x_for_edges_phys,
                  dyn_idx_x_t=dyn_idx_x_t,
                  edge_index_mesh=edge_index,
                  edge_attr_static=edge_attr,
                  use_edge_dyn=USE_EDGE_DYN,
                  use_world_edges=USE_WORLD_EDGES,
                  r_world=R_WORLD,
                  edge_geom_scaler=edge_geom_scaler,
                  edge_dyn_scale=EDGE_DYN_SCALE,
              )

              if k == 0: h = None
              delta_norm, h = model(x_in, ei_for_model, edge_in, h)
              delta_phys = delta_norm * ds_d + dm_d
              y_hat_phys = x_t_phys[:, dyn_idx_x_t] + delta_phys

              # pérdidas (igual que antes)
              loss  = smooth_edge_penalty(y_hat_phys, y_real_phys[t0+k], edge_index, lam_smooth, bc_mask, rigid_mask)
              expanded_mask_free = mask_free.unsqueeze(-1).expand_as(y_hat_phys) if mask_free is not None else None
              loss += masked_smooth_l1_weighted(
                  y_hat_phys, y_real_phys[t0+k],
                  feat_weights=feat_w, beta=SMOOTHL1_BETA, valid_mask=expanded_mask_free
              )
              if bc_mask is not None and bc_mask.any():
                  zero = torch.zeros_like(y_hat_phys[bc_mask][:, :3])
                  loss += lam_bc * F.smooth_l1_loss(y_hat_phys[bc_mask][:, :3], zero, beta=SMOOTHL1_BETA)

              if LAM_GEO > 0:
                  loss += LAM_GEO * edge_geometric_loss(
                      y_hat_phys[:,:3], y_real_phys[t0+k][:,:3], pos0=pos0, edge_index=edge_index,
                      bc_mask=bc_mask, solid_id=rigid_mask, ignore_intersolid=True,
                      lam_vec=LAM_VEC, lam_norm=LAM_NORM, lam_ang=LAM_ANG, relative=True, beta=SMOOTHL1_BETA
                  )
              if LAM_KIN > 0:
                    if len(VEL_DIMS) > 0 and (max(VEL_DIMS) < y_hat_phys.size(1)):
                        disp_prev = x_t_phys[:, dyn_idx_x_t][:, DISP_DIMS]
                        disp_next = y_hat_phys[:, DISP_DIMS]
                        vel_next  = y_hat_phys[:, VEL_DIMS]
                        vel_from_disp = (disp_next - disp_prev) / DT
                        loss += LAM_KIN * F.smooth_l1_loss(vel_next, vel_from_disp, beta=SMOOTHL1_BETA)

              loss_accum += loss

              # --- guardar para métricas ---
              pred_steps.append(y_hat_phys.detach().cpu())
              real_steps.append(y_real_phys[t0+k].detach().cpu())

              # avance autoregresivo
              x_t_phys = x_t_phys.clone()
              x_t_phys[:, dyn_idx_x_t] = y_hat_phys
              x_for_edges_phys = x_t_phys.clone()

              # detector primario de NaN/Inf
              if not torch.isfinite(y_hat_phys).all():
                  exploded_naninf = True

        loss_mean = (loss_accum / float(K)).item()
        total_loss  += loss_mean * N
        total_nodes += N

        # ---- MÉTRICAS ADE/FDE por simulación ----
        y_pred_seq = torch.stack(pred_steps, dim=0)   # (K, N, Ddyn)
        y_real_seq = torch.stack(real_steps, dim=0)   # (K, N, Ddyn)

        m_all = compute_metrics(y_pred_seq, y_real_seq, only_displacement=False)
        m_disp = compute_metrics(y_pred_seq, y_real_seq, only_displacement=True, disp_dims=(0,1,2))

        ADE_all_sum  += m_all["ADE"];  FDE_all_sum  += m_all["FDE"]
        ADE_disp_sum += m_disp["ADE"]; FDE_disp_sum += m_disp["FDE"]
        n_sims_acc   += 1

        # ---- Curva de error por timestep (desplazamiento) ----
        diff = (y_pred_seq[..., :3] - y_real_seq[..., :3])                # (K,N,3)
        l2_t_nodes = torch.norm(diff, dim=-1)                             # (K,N)
        ade_t = l2_t_nodes.mean(dim=1)                                    # (K,)
        per_t_ade_disp_all.append(ade_t)

        # Violación en BC (máximo módulo de disp en nodos fijos, en cualquier t)
        if bc_mask is not None and bc_mask.any():
            disp_bc = y_pred_seq[:, bc_mask.cpu().numpy(), :3]            # (K, N_bc, 3)
            mag_bc = torch.norm(disp_bc, dim=-1)                          # (K, N_bc)
            bc_violation_max_all.append(mag_bc.max().item())
        else:
            bc_violation_max_all.append(0.0)

        if use_tqdm:
            pbar.set_postfix(val=f"{(total_loss/max(total_nodes,1)):.4f}",
                             ADE=f"{m_disp['ADE']:.3f}", FDE=f"{m_disp['FDE']:.3f}",
                             nan=("Y" if exploded_naninf else "N"))

        if exploded_naninf:
            nan_infs += 1

    val_avg = total_loss / max(total_nodes, 1)

    if not return_metrics:
        return val_avg

    # ---- Agregados globales (promedio por simulación) ----
    ADE_all  = ADE_all_sum  / max(n_sims_acc, 1)
    FDE_all  = FDE_all_sum  / max(n_sims_acc, 1)
    ADE_disp = ADE_disp_sum / max(n_sims_acc, 1)
    FDE_disp = FDE_disp_sum / max(n_sims_acc, 1)

    # Curva media de ADE_t (desplaz.) y últimas estadísticas
    if per_t_ade_disp_all:
        # alinear longitudes por si alguna sim tuvo K distinto (aquí debería ser igual)
        K_common = min([len(x) for x in per_t_ade_disp_all])
        per_t_mat = torch.stack([x[:K_common] for x in per_t_ade_disp_all], dim=0)  # (Sims, K)
        ade_t_mean = per_t_mat.mean(dim=0)      # (K,)
        ade_t_p95  = torch.quantile(per_t_mat, 0.95, dim=0)  # (K,)
        # heurística de “estallido”: ratio final / inicial
        growth_ratio = (ade_t_mean[-1] / (ade_t_mean[0].clamp_min(1e-12))).item()
    else:
        ade_t_mean = torch.tensor([])
        ade_t_p95  = torch.tensor([])
        growth_ratio = 1.0

    metrics = {
        "val_loss": val_avg,
        "ADE_all": ADE_all, "FDE_all": FDE_all,
        "ADE_disp": ADE_disp, "FDE_disp": FDE_disp,
        "ADEt_mean": ade_t_mean,            # tensor (K,)
        "ADEt_p95": ade_t_p95,              # tensor (K,)
        "growth_ratio_last_over_first": growth_ratio,
        "bc_violation_max_mean": float(sum(bc_violation_max_all)/max(len(bc_violation_max_all),1)),
        "nan_inf_sims": nan_infs,
        "num_sims": n_sims_acc,
    }
    return val_avg, metrics


## rollout

In [ ]:
@torch.no_grad()
def rollout(
    model,
    T_eff: int,
    x0: torch.Tensor,                 # (N, Din) EN ESPACIO FÍSICO (desnormalizado)
    pos0: torch.Tensor,               # (N, 3) EN ESPACIO FÍSICO (desnormalizado)
    dyn_idx_x,                          # lista/LongTensor de índices dinámicos en x
    edge_index,
    edge_attr,
    x_scaler,                         # (x_mean, x_std) de Din cols
    delta_scaler,
    edge_scaler,
    edge_geom_scaler,
    pos_scaler,
    bc_mask=None,                     # Bool [N] (opcional)
    clamp_bc=False,                   # si quieres forzar 0 en desplazamientos de nodos fijos
    device='cuda',
    return_full=False                 # True -> devuelve la secuencia de x_t completas; False -> sólo y_hat por paso
):
    """
    x0: estado inicial con TODAS las columnas de entrada del modelo (Din).
        Si alguna estática no la tienes en x0, añádela antes.
    dyn_idx: posiciones en x que el modelo predice y que se actualizan en cada paso.
    disp_idx_in_dyn_x: subset dentro de las dinámicas que corresponde a desplazamientos.
    """
    xm, xs = x_scaler
    pm, ps = pos_scaler
    xm_d, xs_d = xm.to(device), xs.to(device)
    pm_d, ps_d = pm.to(device), ps.to(device)

    dm, ds = delta_scaler  # NUEVO
    dm_d, ds_d = dm.to(device), ds.to(device)  # NUEVO

    x_t = x0.to(device)                              # (N, Din)


    # print(f"Initial : x_t{x_t[:10,:]}")
    edge_index = edge_index.to(device)
    if edge_attr is not None and edge_scaler is not None:
          edge_attr = (edge_attr - edge_scaler[0]) / edge_scaler[1]
    edge_attr  = edge_attr.to(device) if edge_attr is not None else None

    if bc_mask is not None:
        bc_mask = bc_mask.to(device).bool()

    if isinstance(dyn_idx_x, (list, tuple)):
        dyn_idx_x_t = torch.as_tensor(dyn_idx_x, device=device)
    else:
        dyn_idx_x_t = dyn_idx_x
    preds = []
    states = [x_t.clone()]

    x_for_edges_phys = x_t.clone()

    with torch.no_grad():
      for i in range(T_eff - 1):

          x_in_norm = (x_t - xm_d) / xs_d
          x_in = None
          if AUGMENT_GEOM_NODE_FEATURES:
            x_in = torch.cat([x_in_norm, create_geom_node_features(x_t, dyn_idx_x_t, pos0, pos_scaler, device, POS_NORM_MODE, POS_SCALE)], dim=1)
          else:
            x_in = x_in_norm


          ei_for_model, edge_in = make_dynamic_edge_inputs(
                      pos0=pos0,
                      x_for_edges_phys=x_for_edges_phys,     # track pred-only
                      dyn_idx_x_t=dyn_idx_x_t,
                      edge_index_mesh=edge_index,
                      edge_attr_static=edge_attr,            # estáticos (normalizados si procede)
                      use_edge_dyn=USE_EDGE_DYN,
                      use_world_edges=USE_WORLD_EDGES,       # << tu flag nuevo
                      r_world=R_WORLD,                       # hiperparámetro
                      edge_geom_scaler=edge_geom_scaler,
                      edge_dyn_scale=EDGE_DYN_SCALE,
                  )
          # edge_in = None
          # if USE_EDGE_DYN:
          #     edge_in = create_geom_edge_features(x_for_edges_phys=x_for_edges_phys,dyn_idx_x_t=dyn_idx_x_t, pos0=pos0, edge_index=edge_index,  edge_geom_scaler=edge_geom_scaler, edge_attr=edge_attr, edge_dyn_scale=EDGE_DYN_SCALE)
          # else:
          #     edge_in = edge_attr  # sólo estáticos si existen

          # normalized output
          if i == 0: h = None
          delta_norm, h  = model(x_in, ei_for_model, edge_in, h)      # (N, Dout)
          # delta_norm  = model(x_in, edge_index, edge_in)      # (N, Dout)
          # delta_phys = delta_norm * ys_d + ym_d   # (N, Dout)
          delta_phys = delta_norm * ds.to(delta_norm) + dm.to(delta_norm)


          y_next_phys = x_t[:, dyn_idx_x_t] + delta_phys   # (N, D_out)

          # clamp a 0 en desplazamientos de nodos fijos (si aplica)
          # TODO: Implement
          if clamp_bc and bc_mask is not None and dyn_idx_x is not None:
              # y_hat[bc_mask, disp_idx_in_dyn] = 0
              # como y_hat es (N,Dout), indexa filas y columnas:
              y_next_phys[bc_mask, 0:3] = 0.0   # asumiendo [dx, dy, dz] en 0..2

          # actualiza x_t SOLO en las columnas dinámicas
          x_t = x_t.clone()
          x_t[:, dyn_idx_x_t] = y_next_phys
          x_for_edges_phys = x_t.clone()

          preds.append(y_next_phys)
          if return_full:
              states.append(x_t.clone())

    return (torch.stack(states, 0) if return_full else torch.stack(preds, 0))

## Scheduling functions

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR, CyclicLR




def k_schedule(epoch, max_epochs, kmin_start=1, kmin_end=2, kmax_start=2, kmax_end=16):
    # sube linealmente desde (kmin_start, kmax_start) hasta (kmin_end, kmax_end)
    p = min(1.0, epoch / max_epochs)
    k_min = int(round(kmin_start + p*(kmin_end - kmin_start)))
    k_max = int(round(kmax_start + p*(kmax_end - kmax_start)))
    k_min = max(1, min(k_min, k_max))
    return k_min, k_max

# Longitud de ciclo CLR (en steps); aquí tomamos 1 ciclo = 1 época
def clr_steps_for_epoch(num_train_iters_per_epoch: int):
    # un triángulo por época
    step_up   = max(1, num_train_iters_per_epoch // 2)
    step_down = max(1, num_train_iters_per_epoch - step_up)
    return step_up, step_down

# ------------------ Helpers ------------------
def make_optimizer_and_scheduler(model, base_lr, max_lr, steps_up, steps_down):
    opt = optim.Adam(model.parameters(), lr=base_lr, weight_decay=WEIGHT_DECAY)
    # Sin momentum en AdamW -> cycle_momentum=False
    sch = CyclicLR(opt, base_lr=base_lr, max_lr=max_lr,
                   step_size_up=steps_up, step_size_down=steps_down,
                   mode='triangular2', cycle_momentum=False)
    return opt, sch
def make_optimizer_and_scheduler_one_cycle(model, max_lr, base_lr, total_steps):
    opt = optim.Adam(model.parameters(), lr=max_lr, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=max_lr,                 # sube el pico para explorar (3×)
        total_steps=total_steps,
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=max_lr/base_lr,             # base_lr ≈ 1.2e-4
        cycle_momentum=False
    )
    return opt, sch

In [ ]:
def save_checkpoint(path, model, x_scaler, y_scaler, delta_scaler, edge_scaler):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(),
                "x_scaler": x_scaler, "y_scaler": y_scaler,"delta_scaler": delta_scaler, "pos_scaler": pos_scaler, "edge_scaler": edge_scaler}, path)
    print("Saved best checkpoint ->", path)

In [ ]:
def check_vram(threshold=0.9):
    total = torch.cuda.get_device_properties(0).total_memory
    allocated = torch.cuda.memory_allocated(0)
    if allocated / total > threshold:
        print("⚠️ Cerca del límite de VRAM, reduciendo batch/K o parando entrenamiento.")
        return True
    return False

def mem():
    a = torch.cuda.memory_allocated() / 1024**2
    r = torch.cuda.memory_reserved() / 1024**2
    return a, r

In [ ]:
# pip install imageio imageio-ffmpeg

import numpy as np
import torch
import pyvista as pv
import imageio
from tqdm import trange

def unique_undirected_edges(edge_index: torch.Tensor):
    ei = edge_index.detach().cpu().numpy().T
    undirected = set()
    for u, v in ei:
        if u == v: continue
        a, b = (u, v) if u < v else (v, u)
        undirected.add((a, b))
    return np.array(list(undirected), dtype=np.int64)

def _build_vtk_lines(edges_uv: np.ndarray) -> np.ndarray:
    e = edges_uv.astype(np.int64, copy=False)
    counts = np.full((e.shape[0], 1), 2, dtype=np.int64)
    return np.hstack([counts, e]).ravel()

def animate_simulation_vtk(
    sim_info: dict,
    model=None, dyn_idx_x=None,
    x_scaler=None, delta_scaler= None, edge_scaler=None, edge_geom_scaler = None, pos_scaler = None,
    device="cuda",
    save_path="rollout_vtk.mp4",
    max_edges=4000, stride=1, framerate=15,
    window_size=(1280, 720),
    show_points=True, point_size=3.0,
    show_bc=True, bc_point_size=9.0,
    tube_lines=False, line_width=1.0,
    camera="iso",
):
    # -------- Datos --------
    pos0       = sim_info["pos0"]
    edge_index = sim_info["edge_index"]
    y_real     = sim_info["y_real"]
    x0         = sim_info["x0"]
    T_eff      = sim_info["T_eff"]

    bc_mask    = sim_info.get("bc_mask", None)
    bc_m = bc_mask.to(device).bool() if bc_mask is not None else None
    rigid_mask    = sim_info.get("rigid_mask", None)
    fixed_idx  = sim_info.get("fixed_idx", None)

    edge_attr = sim_info.get("edge_attr", None)

    # Predicción (si no viene precomputada)
    if model is not None:
        model.eval()
        with torch.no_grad():
            pred = rollout(model, T_eff, x0, pos0, dyn_idx_x, edge_index, edge_attr, x_scaler, delta_scaler,edge_scaler, edge_geom_scaler, pos_scaler, bc_mask, CLAMP_BC_IN_ROLLOUT, device, False)
    else:
        pred = sim_info["y_pred"]

    for p in pred:
      assert (p[bc_m, :3].abs().max() < 1e-8), "BC con Δ != 0"
    # -------- A NumPy --------
    pos0_np = pos0.detach().cpu().numpy()
    gt_np   = y_real.detach().cpu().numpy()[..., :3]
    pr_np   = pred.detach().cpu().numpy()[..., :3]
    Tm1, N, _ = gt_np.shape

    # -------- Edges --------
    edges_uv = unique_undirected_edges(edge_index)
    edges_uv = edges_uv.detach().cpu().numpy() if isinstance(edges_uv, torch.Tensor) else edges_uv
    if max_edges is not None and len(edges_uv) > max_edges:
        rng = np.random.RandomState(0)
        edges_uv = edges_uv[rng.choice(len(edges_uv), size=max_edges, replace=False)]
    vtk_lines = _build_vtk_lines(edges_uv)

    # -------- Rango espacial (solo para cámara) --------
    all_gt = pos0_np[None, ...] + gt_np
    all_pr = pos0_np[None, ...] + pr_np
    xyz_min = np.minimum(all_gt.min(axis=(0, 1)), all_pr.min(axis=(0, 1)))
    xyz_max = np.maximum(all_gt.max(axis=(0, 1)), all_pr.max(axis=(0, 1)))
    pad = 0.05 * (xyz_max - xyz_min + 1e-12)
    xyz_min -= pad; xyz_max += pad
    center = (xyz_min + xyz_max) / 2.0

    # -------- Dos plotters off-screen (mismo tamaño) --------
    single_size = (max(1, window_size[0] // 2), window_size[1])
    pv.global_theme.window_size = single_size
    pv.global_theme.anti_aliasing = "ssaa"
    pv.global_theme.smooth_shading = False
    pv.global_theme.background = "white"

    # Coloreado
    rigid_np = None
    rgb_colors = None
    if rigid_mask is not None:
        rigid_np = rigid_mask.detach().cpu().numpy().astype(np.int32).reshape(-1)
        rgb_colors = np.zeros((rigid_np.shape[0], 3), dtype=np.uint8)
        rgb_colors[rigid_np == 0] = (0, 255, 0)     # verde
        rgb_colors[rigid_np != 0] = (255, 0, 0)     # rojo

    # === GT ===
    pl_gt = pv.Plotter(off_screen=True, window_size=single_size)
    pl_gt.set_background("white")
    pl_gt.add_text("Ground Truth", font_size=14)
    mesh_gt = pv.PolyData(pos0_np + gt_np[0], lines=vtk_lines)
    if rgb_colors is not None:
      mesh_gt["rgb"] = rgb_colors
      pl_gt.add_mesh(
          mesh_gt, scalars="rgb", rgb=True, line_width=line_width,
          style="wireframe", lighting=False, render_lines_as_tubes=tube_lines,
          opacity=0.9, smooth_shading=False
      )
    else:
        pl_gt.add_mesh(
            mesh_gt, color="seagreen", line_width=line_width,
            style="wireframe", lighting=False, render_lines_as_tubes=tube_lines,
            opacity=0.9, smooth_shading=False
        )
    pts_gt = pts_gt_free = pts_gt_fix = None
    if show_points:
        if show_bc and bc_mask is not None:
            bc_np = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np
            pts_gt_free = pv.PolyData((pos0_np + gt_np[0])[free_np])
            pts_gt_fix  = pv.PolyData((pos0_np + gt_np[0])[bc_np])
            pl_gt.add_mesh(pts_gt_free, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="seagreen", opacity=0.9)
            pl_gt.add_mesh(pts_gt_fix,  style="points", point_size=bc_point_size,
                           render_points_as_spheres=True, color="gold", opacity=1.0)
        else:
            pts_gt = pv.PolyData(pos0_np + gt_np[0])
            pl_gt.add_mesh(pts_gt, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="seagreen", opacity=0.9)
    if camera == "iso":
        pl_gt.view_isometric()
    pl_gt.show_bounds(xtitle="x", ytitle="y", ztitle="z")
    pl_gt.show_axes()
    pl_gt.reset_camera()
    pl_gt.reset_camera_clipping_range()

    # === Pred ===
    pl_pr = pv.Plotter(off_screen=True, window_size=single_size)
    pl_pr.set_background("white")
    pl_pr.add_text("Prediction", font_size=14)
    mesh_pr = pv.PolyData(pos0_np + pr_np[0], lines=vtk_lines)
    if rgb_colors is not None:
      mesh_pr["rgb"] = rgb_colors
      pl_pr.add_mesh(
          mesh_pr, scalars="rgb", rgb=True, line_width=line_width,
          style="wireframe", lighting=False, render_lines_as_tubes=tube_lines,
          opacity=0.9, smooth_shading=False
      )
    else:
        pl_pr.add_mesh(
            mesh_pr, color="crimson", line_width=line_width,
            style="wireframe", lighting=False, render_lines_as_tubes=tube_lines,
            opacity=0.9, smooth_shading=False
        )
    pts_pr = pts_pr_free = pts_pr_fix = None
    if show_points:
        if show_bc and bc_mask is not None:
            bc_np = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np
            pts_pr_free = pv.PolyData((pos0_np + pr_np[0])[free_np])
            pts_pr_fix  = pv.PolyData((pos0_np + pr_np[0])[bc_np])
            pl_pr.add_mesh(pts_pr_free, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="crimson", opacity=0.9)
            pl_pr.add_mesh(pts_pr_fix,  style="points", point_size=bc_point_size,
                           render_points_as_spheres=True, color="gold", opacity=1.0)
        else:
            pts_pr = pv.PolyData(pos0_np + pr_np[0])
            pl_pr.add_mesh(pts_pr, style="points", point_size=point_size,
                           render_points_as_spheres=True, color="crimson", opacity=0.9)
    if camera == "iso":
        pl_pr.view_isometric()
    pl_pr.show_bounds(xtitle="x", ytitle="y", ztitle="z")
    pl_pr.show_axes()
    pl_pr.reset_camera()
    pl_pr.reset_camera_clipping_range()

    # -------- Encoder --------
    writer = imageio.get_writer(save_path, fps=framerate, codec="libx264", quality=8)

    # Primer frame
    img_gt = pl_gt.screenshot(return_img=True)
    img_pr = pl_pr.screenshot(return_img=True)
    frame  = np.concatenate([img_gt, img_pr], axis=1)
    writer.append_data(frame)

    # -------- Loop --------
    for t in trange(1, Tm1, desc="Render VTK (SxS)", leave=False):
        if (t % stride) != 0:
            continue

        Pgt = pos0_np + gt_np[t]
        Ppr = pos0_np + pr_np[t]

        # Actualiza geometría in-place
        mesh_gt.points = Pgt
        mesh_pr.points = Ppr

        if show_points and show_bc and bc_mask is not None:
            bc_np = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np
            if pts_gt_free is not None:
                pts_gt_free.points = Pgt[free_np]
                pts_gt_fix.points  = Pgt[bc_np]
            if pts_pr_free is not None:
                pts_pr_free.points = Ppr[free_np]
                pts_pr_fix.points  = Ppr[bc_np]
        elif show_points:
            if pts_gt is not None: pts_gt.points = Pgt
            if pts_pr is not None: pts_pr.points = Ppr

        # Render y captura de cada plotter
        pl_gt.render(); img_gt = pl_gt.screenshot(return_img=True)
        pl_pr.render(); img_pr = pl_pr.screenshot(return_img=True)
        frame = np.concatenate([img_gt, img_pr], axis=1)
        writer.append_data(frame)

    writer.close()
    pl_gt.close(); pl_pr.close()
    return save_path


## Parameters

In [ ]:
# Configuration parameters
SEED_NUMBER = 42
MIN_T = 5
STEP = 1 # To select a smaller number of attributes from the database
PATIENCE = 20
# MAX_EPOCHS = 200
MAX_EPOCHS = 50
FINE_TUNING_EPOCHS = 30
LR_MIN_FINE_TUNING = 1e-5
N_LAYERS= 5
HIDDEN= 256
WEIGHT_DECAY = 2.3465254539958182e-05
DROPOUT=0.09
MAX_GRAD_NORM = 0.8388359885282621

# Loss weights
LAM_BC = 5E-3
LAM_SMOOTH = 0.012753330601120494
LAM_KIN = 0.012981351206458915        # peso consistencia cinemática (si tienes velocidades)
# LAM_GEO = 0.033372913516991586
LAM_GEO = 0.083372913516991586
LAM_VEC=0.5170142513796345
LAM_NORM=0.05714986186823738
# LAM_ANG=0.24105233141330312
LAM_ANG=0.64105233141330312
LAM_RIGID = 1e-3


CLAMP_BC_IN_ROLLOUT = True

REAL_TIME_STEP = 0.0001504529 * 5
SMOOTHL1_BETA = 1.0      # 1.0 (default PyTorch). Baja a 0.5 si quieres más robustez.
DT = 0.0001504529 * 5                 # tamaño de paso temporal (ajusta a tu dataset)
DISP_DIMS = [0,1,2]
VEL_DIMS  = [3,4,5]      # déjalo [] si aún no metes velocidades

AUGMENT_GEOM_NODE_FEATURES = True  # añade pos_t (x,y,z) a la entrada del modelo
POS_NORM_MODE = "dataset"
POS_SCALE = 0.03745928442366847

# Dynamic edge attributes
USE_EDGE_DYN = True
EDGE_DYN_NORM_MODE = "dataset"
EDGE_DYN_SCALE = 1.9803683902320412e-05
USE_WORLD_EDGES = True
R_WORLD = 12

USE_INPUT_NOISE = True          # activar inyección de ruido
NOISE_WARMUP_EPOCHS = 5        # cuántas épocas para subir σ
SIGMA_DISP_MAX_STD = 0.25       # σ_disp = 0.25 * std(delta_disp)  (ajústalo)
SIGMA_VEL_MAX_STD  = 0.15       # σ_vel  = 0.15 * std(delta_vel)   (ajústalo)
VEL_FROM_DISP = True            # v-noise coherente con Δx/dt si no se fija sigma_vel
DT_TRAIN = 0.0001504529 * 5                 # dt físico (si lo tienes)

# p_teacher
P_TEACHER_P0= 0.8
P_TEACHER_PMIN = 0.05
P_TEACHER_WARMUP = 15

K_MIN_START = 1
K_MIN_END = 2
K_MAX_START = 10
K_MAX_END = 100
K_EVAL=K_MAX_END

LR_MIN = 4.952926387339979e-05
LR_MAX = 0.0010901907508582193


# ATTRIBUTES_WEIGHTS = [2.0, 2.0, 2.0, 1, 1, 1]
# ATTRIBUTES_WEIGHTS = [2.0, 2.0, 2.0, 0.5, 0.5, 0.5]
DISP_WEIGHT = 0.8890076506112339
VEL_WEIGHT = 0.7159661412516282
ATTRIBUTES_WEIGHTS = [DISP_WEIGHT, DISP_WEIGHT, DISP_WEIGHT, VEL_WEIGHT, VEL_WEIGHT, VEL_WEIGHT]

set_seed(SEED_NUMBER)

INPUT_DIR="/content/drive/MyDrive/CrashGeoNN/graphs_iteration_3/"

# Attributes [Delta_x, Delta_y, Delta_z, V_x, V_y, V_z, A_z, A_y, A_z, bc_mask, rigid_mask]
X_DISCARD_INDEX = [6,7,8] # We discard accelerations
X_DYNAMIC_INDEX = [0,1,2,3,4,5]
X_STATIC_INDEX = [6,7] # After removing the previous index bc_mask and rigid_mask remain
Y_DISCARD_INDEX = [6,7,8] # We discard accelerations
Y_DYNAMIC_INDEX = [0,1,2,3,4,5]
Y_STATIC_INDEX = [] # No static attributes here






In [ ]:
# === Cambia esta ruta a tu .pt (Drive o local) ===
DB_PATH = INPUT_DIR  # p.ej.: "/content/drive/MyDrive/Crash-GeoNN/GRAPHS.pt"
simulations = load_database_dir(DB_PATH, STEP)


Loading DB from: /content/drive/MyDrive/CrashGeoNN/graphs_iteration_3/
Found 93 graphs


  0%|          | 0/93 [00:00<?, ?it/s]

[sim 0] N=3544 E=28108 undirected? True duplicates? False
[sim 1] N=3568 E=28310 undirected? True duplicates? False
[sim 2] N=3499 E=27772 undirected? True duplicates? False
[sim 3] N=3554 E=28208 undirected? True duplicates? False
[sim 4] N=3546 E=28152 undirected? True duplicates? False


In [ ]:
print(f"Number of features before: {simulations[0][0].x.shape[1]}")
simulations = drop_features_db(simulations,X_DISCARD_INDEX, Y_DISCARD_INDEX)
INPUT_FEATURES = simulations[0][0].x.shape[1]
OUTPUT_FEATURES = simulations[0][0].y.shape[1]
print(f"Final number of INPUT features: {INPUT_FEATURES}. Dynamic: {len(X_DYNAMIC_INDEX)}. Static: {len(X_STATIC_INDEX)}")
assert INPUT_FEATURES == (len(X_DYNAMIC_INDEX) + len(X_STATIC_INDEX))
print(f"Final number of OUTPUT features: {OUTPUT_FEATURES}. Dynamic: {len(Y_DYNAMIC_INDEX)}. Static: {len(Y_STATIC_INDEX)}")
assert OUTPUT_FEATURES == (len(Y_DYNAMIC_INDEX) + len(Y_STATIC_INDEX))

assert len(X_DYNAMIC_INDEX) == len(Y_DYNAMIC_INDEX) # Needed for the rollout

if AUGMENT_GEOM_NODE_FEATURES:
  INPUT_FEATURES += 3 # x,y,z



all_ids = list(range(len(simulations)))
train_ids, val_ids, test_ids = split_simulations(all_ids, train_ratio=0.7, val_ratio=0.15, seed=42)

Number of features before: 11
Final number of INPUT features: 8. Dynamic: 6. Static: 2
Final number of OUTPUT features: 6. Dynamic: 6. Static: 0


In [ ]:
print("Building splits...")
train_graphs, train_static = build_split_from_db(simulations, train_ids, min_time_step=MIN_T)
val_graphs,   val_static   = build_split_from_db(simulations, val_ids, min_time_step=MIN_T)
test_graphs,  test_static  = build_split_from_db(simulations, test_ids, min_time_step=MIN_T)

edge_dim = train_graphs[0].edge_attr.size(1) if hasattr(train_graphs[0], "edge_attr") and train_graphs[0].edge_attr is not None else 0
assert edge_dim > 0, "edge_attr required for GINEConv; si no tienes, cambia a un modelo sin edge_attr."

if USE_EDGE_DYN:
    edge_dim += 4


Building splits...


In [ ]:
# Fit (con prints)
x_scaler, y_scaler = fit_scaler(train_graphs, X_DYNAMIC_INDEX, Y_DYNAMIC_INDEX, verbose=True)
delta_scaler       = fit_delta_scaler(train_graphs, X_DYNAMIC_INDEX, Y_DYNAMIC_INDEX, verbose=True)
pos_scaler         = fit_pos_scaler(train_static, verbose=True)
edge_geom_scaler   = fit_edge_geom_scaler(train_static, verbose=True)

# Apply (con prints rápidos)
apply_scaler(train_graphs, x_scaler, y_scaler, verbose=True)
apply_scaler(val_graphs,   x_scaler, y_scaler, verbose=True)
apply_scaler(test_graphs,  x_scaler, y_scaler, verbose=True)

edge_scaler = fit_edge_attr_scaler(train_graphs, verbose=True)
apply_edge_attr_scaler(train_graphs, edge_scaler, verbose=True)
apply_edge_attr_scaler(val_graphs,   edge_scaler, verbose=True)
apply_edge_attr_scaler(test_graphs,  edge_scaler, verbose=True)

# Sanity checks (espera medias ~0 y std ~1 en train; en val/test variará, pero cerca)
sanity_check_attr(train_graphs, "x", X_DYNAMIC_INDEX)
sanity_check_attr(train_graphs, "y", Y_DYNAMIC_INDEX)
if edge_scaler is not None:
    sanity_check_attr(train_graphs, "edge_attr")

[fit_scaler] X: muestras=45017476, cols_afectadas=6, std[min,max]=(5.91,1.15e+03)
[fit_scaler] Y: muestras=45017476, cols_afectadas=6, std[min,max]=(5.92,1.15e+03)
[fit_delta_scaler] muestras=45017476, C=6, std[min,max]=(0.037,27.8)
[fit_pos_scaler] muestras=229681, C=3, std[min,max]=(77,321)
[fit_edge_geom_scaler] muestras=1823314, C=4, std[min,max]=(5.82,13.2)
[apply_scaler] ejemplo X: mean(abs)=0.408, std(mean)=0.76
[apply_scaler] ejemplo Y: mean(abs)=0.498, std(mean)=0.905
[apply_scaler] grafos procesados=12740
[apply_scaler] ejemplo X: mean(abs)=0.422, std(mean)=0.768
[apply_scaler] ejemplo Y: mean(abs)=0.515, std(mean)=0.904
[apply_scaler] grafos procesados=2548
[apply_scaler] ejemplo X: mean(abs)=0.412, std(mean)=0.771
[apply_scaler] ejemplo Y: mean(abs)=0.502, std(mean)=0.911
[apply_scaler] grafos procesados=2940
[fit_edge_attr_scaler] muestras=357369544, C=4, std[min,max]=(0.467,5.82)
[apply_edge_attr_scaler] grafos con edge_attr normalizado=12740
[apply_edge_attr_scaler] graf

## Optuna hyperparameter optimizacion

### Dictionary creation

In [ ]:
def suggest_hparams(trial: optuna.Trial):
    h = {}
    # Optimización
    h["weight_decay"]= trial.suggest_float("weight_decay", 0.0, 5e-4, log=False)
    # Arquitectura
    h["layers"]      = trial.suggest_int("layers", 3, 15)
    h["hidden"]      = trial.suggest_int("hidden", 64, 512, step=32)
    h["dropout"]     = trial.suggest_float("dropout", 0.0, 0.5)

    # Free-run / exposición al error
    h["lam_bc"]      = trial.suggest_float("lam_bc", 1e-3, 1e-1, log=True)
    h["lam_smooth"]  = trial.suggest_float("lam_smooth", 1e-5, 1e-1, log=True)
    h["lam_kin"]     = trial.suggest_float("lam_kin", 1e-5, 1e-1, log=True)
    h["lam_geo"]     = trial.suggest_float("lam_geo", 1e-5, 1e-1, log=True)
    h["edge_dyn_scale"] = trial.suggest_float("edge_dyn_scale", 1e-5, 1e-1, log=True)

    # Options
    h["pos_scale"] = trial.suggest_float("pos_scale", 0.0, 1.0)

    h["edge_dyn_scale"] = trial.suggest_float("edge_dyn_scale", 1e-5, 1e-1, log=True)
    h["r_world"] = trial.suggest_int("r_world", 3, 15)

    h["disp_weights"] = trial.suggest_float("disp_weights", 0.5, 1.0)
    h["vel_weights"] = trial.suggest_float("vel_weights", 0.1, 1.0)
    h["lam_vec"] = trial.suggest_float("lam_vec", 0.0, 1.0)
    h["lam_norm"] = trial.suggest_float("lam_norm", 0.0, 1.0)
    h["lam_ang"] = trial.suggest_float("lam_ang", 0.0, 1.0)

    h["max_grad_norm"] = trial.suggest_float("max_grad_norm", 0.5, 1.0)


    return h

In [ ]:
def train_for_trial(h, trial: optuna.Trial):
    LAM_BC = h["lam_bc"]
    LAM_SMOOTH = h["lam_smooth"]
    LAM_KIN = h["lam_kin"]
    LAM_GEO = h["lam_geo"]
    LAM_VEC = h["lam_vec"]
    LAM_NORM = h["lam_norm"]
    LAM_ANG = h["lam_ang"]

    WEIGHT_DECAY = h["weight_decay"]
    EDGE_DYN_SCALE = h["edge_dyn_scale"]

    ATTRIBUTES_WEIGHTS = [h["disp_weights"], h["disp_weights"], h["disp_weights"], h["vel_weights"],h["vel_weights"],h["vel_weights"]]


    POS_SCALE = h["pos_scale"]
    R_WORLD = h["r_world"]

    DROPOUT = h["dropout"]

    model = ImpactGNN_Edge(in_ch=INPUT_FEATURES, edge_attr_dim=edge_dim, hidden=h["hidden"], out_ch=OUTPUT_FEATURES, layers=h["layers"], dropout=DROPOUT).to(device)
    scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))


    MAX_EPOCHS = 25

    opt = optim.Adam(model.parameters(), lr=LR_MIN, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=LR_MAX,                 # sube el pico para explorar (3×)
        total_steps=len(train_static) * MAX_EPOCHS,
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=LR_MAX/LR_MIN,             # base_lr ≈ 1.2e-4
        cycle_momentum=False
    )

    print(f"Training for up to {MAX_EPOCHS} epochs...")
    best_val = float('inf')
    wait = 0
    for epoch in range(1, MAX_EPOCHS+1):

      K_min, K_max =  k_schedule(epoch, MAX_EPOCHS, K_MIN_START, K_MIN_END, K_MAX_START, K_MAX_END)

      train_loss = train_epoch_k(model, train_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, edge_geom_scaler, pos_scaler, device, epoch, LAM_SMOOTH, LAM_BC, K_min, K_max, scaler, opt, MAX_GRAD_NORM, ATTRIBUTES_WEIGHTS, scheduler=sch)
      val_loss, m = eval_epoch_k(model, val_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, edge_geom_scaler, pos_scaler, device, LAM_SMOOTH, LAM_BC, 180, ATTRIBUTES_WEIGHTS, return_metrics=True)
      print(
          f"[ val_loss={m['val_loss']:.4f} | "
          f"ADEd={m['ADE_disp']:.4f} FDEd={m['FDE_disp']:.4f} | "
          f"grow={m['growth_ratio_last_over_first']:.2f} | "
          f"bc_max={m['bc_violation_max_mean']:.4e} | "
          f"NaN_sims={m['nan_inf_sims']}/{m['num_sims']}"
      )
      fde = m['FDE_disp']
      print(f"[Epoch {epoch:03d}](LR= {opt.param_groups[0]['lr']:.2e}), (Kmax: {K_max}) train {train_loss:.6f}, val {val_loss:.6f}, FDE {fde:.6f}")
      trial.report(fde, step=epoch)
      if trial.should_prune():
            raise optuna.TrialPruned()
      if fde + 1e-6 < best_val:
        best_val = fde
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}


    return best_val


In [ ]:
def objective(trial: optuna.Trial):
    h = suggest_hparams(trial)

    try:
        best_fde = train_for_trial(h, trial)
        return best_fde  # Optuna lo minimiza
    except RuntimeError as e:
        # Si fallan kernels no deterministas o falta de memoria, prunea
        if "out of memory" in str(e).lower() or "deterministic" in str(e).lower():
            raise optuna.TrialPruned()
        raise
def optimize():

    storage_url = "sqlite:///optuna_study.db"
    study = optuna.create_study(
        study_name="gnn_impact_optimization_v2",
        direction="minimize",
        sampler=TPESampler(seed=2024, n_startup_trials=10),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
        storage=storage_url,
        load_if_exists=True,
    )

    # Número de intentos (ajusta según tu tiempo/cómputo)
    study.optimize(objective, n_trials=50, gc_after_trial=True)

    print("== Mejores parámetros ==")
    print(study.best_params)
    print("Mejor FDE:", study.best_value)

    # (Opcional) obtener top-N para reentrenar más largo
    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else math.inf)[:5]
    for i, t in enumerate(top_trials, 1):
        print(f"[Top {i}] value={t.value:.5f} params={t.params}")



In [ ]:
optimizeHyperparameters = False

if optimizeHyperparameters:
  optimize()

## Training block

In [ ]:
import gc, torch
# model = ImpactGNN_Edge(in_ch=INPUT_FEATURES, edge_attr_dim=edge_dim, hidden=HIDDEN, out_ch=OUTPUT_FEATURES, layers=N_LAYERS, dropout=DROPOUT).to(device)
model = ImpactGNN_EdgeGRU(in_ch=INPUT_FEATURES,
                          edge_attr_dim=edge_dim,
                          hidden=HIDDEN, out_ch=len(Y_DYNAMIC_INDEX),
                          layers=N_LAYERS, dropout=DROPOUT).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

opt = optim.AdamW(model.parameters(), lr=LR_MIN, weight_decay=WEIGHT_DECAY)
# sch = torch.optim.lr_scheduler.OneCycleLR(
#     opt,
#     max_lr=LR_MAX,                 # sube el pico para explorar (3×)
#     total_steps=len(train_static) * MAX_EPOCHS,
#     pct_start=0.3,
#     anneal_strategy='cos',
#     div_factor=LR_MAX/LR_MIN,             # base_lr ≈ 1.2e-4
#     cycle_momentum=False
# )
num_train_iters = max(1, len(train_static))   # 1 paso por sim (porque mezclas simulaciones dentro)
steps_up, steps_down = clr_steps_for_epoch(num_train_iters)
sch = CyclicLR(opt, base_lr=LR_MIN, max_lr=LR_MAX,step_size_up=steps_up, step_size_down=steps_down, mode='triangular2', cycle_momentum=False)

PATIENCE  = MAX_EPOCHS

print(f"Training for up to {MAX_EPOCHS} epochs...")
best_val = float('inf')
wait = 0


prevK = 0
for epoch in range(1, MAX_EPOCHS+1):
  if check_vram():
    break
  a, r = mem()
  # print(f"Allocated={a:.0f}MB  Reserved={r:.0f}MB")
  K_min, K_max =  k_schedule(epoch, MAX_EPOCHS, K_MIN_START, K_MIN_END, K_MAX_START, K_MAX_END)

  train_loss = train_epoch_k(model, train_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, edge_geom_scaler, pos_scaler, device, epoch, LAM_SMOOTH, LAM_BC, K_min, K_max, scaler, opt, MAX_GRAD_NORM, ATTRIBUTES_WEIGHTS, scheduler=sch)
  val_loss, m = eval_epoch_k(model, val_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX, edge_scaler, edge_geom_scaler, pos_scaler, device, LAM_SMOOTH, LAM_BC, K_EVAL, ATTRIBUTES_WEIGHTS, return_metrics=True)
  print(
      f"[ val_loss={m['val_loss']:.4f} | "
      f"ADEd={m['ADE_disp']:.4f} FDEd={m['FDE_disp']:.4f} | "
      f"grow={m['growth_ratio_last_over_first']:.2f} | "
      f"bc_max={m['bc_violation_max_mean']:.4e} | "
      f"NaN_sims={m['nan_inf_sims']}/{m['num_sims']}"
  )
  fde = m['FDE_disp']
  print(f"[Epoch {epoch:03d}](LR= {opt.param_groups[0]['lr']:.2e}), (Kmax: {K_max}) train {train_loss:.6f}, val {val_loss:.6f}, FDE {fde:.6f}")
  if fde + 1e-6 < best_val:
    best_val = fde
    best_state = {k: v.cpu() for k,v in model.state_dict().items()}
    wait = 0
    print(f"Saving check point: FDE = {fde}")
  else:
    wait += 1
    if wait >= PATIENCE:
      print(f"Early stopping after {PATIENCE} epochs without improvement")
      break






Training for up to 50 epochs...


Train (epoch 1):   0%|          | 0/65 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Valid:   0%|          | 0/13 [00:00<?, ?it/s]

[ val_loss=1183.5156 | ADEd=48.2655 FDEd=64.2919 | grow=205.98 | bc_max=1.0372e+00 | NaN_sims=0/13
[Epoch 001](LR= 4.95e-05), (Kmax: 12) train 30.381730, val 1183.515636, FDE 64.291922
Saving check point: FDE = 64.29192205575796


...

Valid:   0%|          | 0/13 [00:00<?, ?it/s]

[ val_loss=425.9936 | ADEd=38.0166 FDEd=56.9697 | grow=137.76 | bc_max=9.1288e-01 | NaN_sims=0/13
[Epoch 049](LR= 4.95e-05), (Kmax: 98) train 88.774961, val 425.993614, FDE 56.969674


Train (epoch 50):   0%|          | 0/65 [00:00<?, ?it/s]

Valid:   0%|          | 0/13 [00:00<?, ?it/s]

[ val_loss=432.5573 | ADEd=37.9511 FDEd=59.0311 | grow=126.85 | bc_max=1.7975e+00 | NaN_sims=0/13
[Epoch 050](LR= 4.95e-05), (Kmax: 100) train 60.488560, val 432.557293, FDE 59.031061


Fine tuning

In [ ]:
# === Fine-tuning ===
from torch.optim.lr_scheduler import OneCycleLR

# 0) Cargar mejor estado del primer bloque
if best_state is not None:
    model.load_state_dict(best_state)  # ok

print("Fine-tuning (nuevo optimizer + OneCycle)…")

# A) Teacher Forcing alto y sin ruido (pulido estable)
USE_INPUT_NOISE = False
P_TEACHER_WARMUP = FINE_TUNING_EPOCHS      # rampa suave
P_TEACHER_P0     = 1.0                     # empezar con TF=100%
P_TEACHER_PMIN   = 0.8                     # y no bajar de 0.8 en FT

# B) Optimizer NUEVO (no reutilizar estados)
opt = optim.AdamW(model.parameters(),
                  lr=LR_MIN_FINE_TUNING,             # p.ej. 1e-5
                  weight_decay=WEIGHT_DECAY/10.0)    # ya lo hacías, mantenlo

# C) OneCycle un poco más ancho en el pico
steps_per_epoch = len(train_static)  # 1 step por simulación
max_lr_ft = max(LR_MIN_FINE_TUNING*5, min(LR_MIN, LR_MIN_FINE_TUNING*8))  # abre ventana
sch = OneCycleLR(
    opt,
    max_lr=max_lr_ft,
    steps_per_epoch=steps_per_epoch,
    epochs=FINE_TUNING_EPOCHS,
    pct_start=0.25,
    anneal_strategy='cos',
    cycle_momentum=False
)

best_val = best_val  # seguimos comparando con el mejor anterior
wait = 0
for epoch in range(1, FINE_TUNING_EPOCHS+1):
    # D) K corto para afinar per-step al principio del FT
    if epoch <= 8:
        K_min, K_max = 1, 3
    elif epoch <= 16:
        K_min, K_max = 2, 6
    else:
        K_min, K_max = int(K_MIN_END), int(K_MAX_END)

    train_loss = train_epoch_k(
        model, train_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX,
        edge_scaler, edge_geom_scaler, pos_scaler, device, epoch,
        LAM_SMOOTH, LAM_BC, K_min, K_max, scaler, opt, MAX_GRAD_NORM,
        ATTRIBUTES_WEIGHTS, scheduler=sch
    )

    val_loss, m = eval_epoch_k(
        model, val_static, x_scaler, delta_scaler, X_DYNAMIC_INDEX,
        edge_scaler, edge_geom_scaler, pos_scaler, device,
        LAM_SMOOTH, LAM_BC, K_EVAL, ATTRIBUTES_WEIGHTS, return_metrics=True
    )
    fde = m["FDE_disp"]
    print(f"[FT {epoch:03d}] (LR={opt.param_groups[0]['lr']:.2e}) "
          f"train {train_loss:.6f}, val {val_loss:.6f}, FDE {fde:.6f}")

    if fde + 1e-6 < best_val:
        best_val = fde
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        wait = 0
        print(f"Saving check point: FDE = {fde}")
    else:
        wait += 1
        if wait >= FINE_TUNING_EPOCHS:  # evita cortar precoz por PATIENCE global
            break


Fine-tuning (nuevo optimizer + OneCycle)…


Train (epoch 1):   0%|          | 0/65 [00:00<?, ?it/s]

Valid:   0%|          | 0/13 [00:00<?, ?it/s]

[FT 001] (LR=4.08e-06) train 31.210412, val 379.134356, FDE 45.256665


Train (epoch 2):   0%|          | 0/65 [00:00<?, ?it/s]

...

Valid:   0%|          | 0/13 [00:00<?, ?it/s]

[FT 003] (LR=1.86e-05) train 26.217951, val 397.814590, FDE 45.741171


Train (epoch 4):   0%|          | 0/65 [00:00<?, ?it/s]

Valid:   0%|          | 0/13 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Evaluate results

In [ ]:
if best_state is not None: model.load_state_dict(best_state)

print("Evaluating rollout on TEST simulations…")
model.eval()
metrics_all = []
metrics_disp_all = []
for sid, info in test_static.items():
    edge_index = info['edge_index']
    edge_attr  = info.get('edge_attr', None)
    T_eff = info['T_eff']
    y_real = info['y_real'].to(device)
    x0 = info['x0']
    bc_mask = info['bc_mask']
    pos0 = info['pos0']

    pred = rollout(model, T_eff, x0, pos0, X_DYNAMIC_INDEX, edge_index, edge_attr, x_scaler, delta_scaler, edge_scaler, edge_geom_scaler, pos_scaler, bc_mask, CLAMP_BC_IN_ROLLOUT, device, False)
    m = compute_metrics(pred, y_real)
    metrics_all.append(m)
    m_disp = compute_metrics(pred, y_real, True)
    metrics_disp_all.append(m_disp)
    print(f"[SIM {sid}] MAE={m['MAE']:.6f} RMSE={m['RMSE']:.6f} ADE={m['ADE']:.6f} FDE={m['FDE']:.6f}")
    print(f"[DISPLACEMENT: SIM {sid}] MAE={m_disp['MAE']:.6f} RMSE={m_disp['RMSE']:.6f} ADE={m_disp['ADE']:.6f} FDE={m_disp['FDE']:.6f}")


if metrics_all:
    avg = {k: float(np.mean([d[k] for d in metrics_all])) for k in metrics_all[0].keys()}
    print("==== TEST AVERAGE ===="); [print(f"{k}: {v:.6f}") for k,v in avg.items()]
if metrics_disp_all:
    avg = {k: float(np.mean([d[k] for d in metrics_disp_all])) for k in metrics_disp_all[0].keys()}
    print("==== TEST AVERAGE DISPLACEMENT ===="); [print(f"{k}: {v:.6f}") for k,v in avg.items()]


Evaluating rollout on TEST simulations…
[SIM 41] MAE=217.203934 RMSE=493.784882 ADE=919.321716 FDE=802.525208
[DISPLACEMENT: SIM 41] MAE=14.713232 RMSE=31.354837 ADE=34.283447 FDE=90.002670
[SIM 22] MAE=218.715866 RMSE=505.145020 ADE=944.330750 FDE=845.053040
[DISPLACEMENT: SIM 22] MAE=14.175042 RMSE=29.674534 ADE=33.384151 FDE=84.752800
[SIM 19] MAE=219.231049 RMSE=494.211212 ADE=929.217957 FDE=831.484680
[DISPLACEMENT: SIM 19] MAE=14.104659 RMSE=29.432938 ADE=32.804897 FDE=85.941635
[SIM 35] MAE=217.845779 RMSE=506.572266 ADE=940.610291 FDE=821.585999
[DISPLACEMENT: SIM 35] MAE=14.758489 RMSE=31.057827 ADE=34.674835 FDE=86.994347
[SIM 64] MAE=219.695328 RMSE=498.569061 ADE=930.210693 FDE=835.697998
[DISPLACEMENT: SIM 64] MAE=14.372241 RMSE=30.535803 ADE=33.667839 FDE=87.499794
[SIM 47] MAE=221.459290 RMSE=506.885162 ADE=942.701721 FDE=834.927734
[DISPLACEMENT: SIM 47] MAE=14.662710 RMSE=31.081244 ADE=34.504173 FDE=89.909637
[SIM 12] MAE=216.160278 RMSE=493.669830 ADE=919.901367 FDE=8

In [ ]:
print("Creating  test animation...")
sid = next(iter(test_static.keys()))
animation_name = "rollout_test_one_cycle_" + str(sid) + ".mp4"
animation_path = "/content/drive/MyDrive/CrashGeoNN/" + animation_name
animate_simulation_vtk(test_static[sid],model,X_DYNAMIC_INDEX, x_scaler,delta_scaler,edge_scaler, edge_geom_scaler, pos_scaler,device="cuda",save_path=animation_path,
    max_edges=4000,stride=1,framerate=15,window_size=(1280, 720),show_points=True,point_size=3.0,show_bc=True,bc_point_size=9.0,
    tube_lines=False,          # True = tubos 3D (más bonito, algo más lento)line_width=1.0,
    camera="iso")

print("Creating  train animation...")
sid = next(iter(train_static.keys()))
animation_name = "rollout_train_one_cycle_" + str(sid) + ".mp4"
animation_path = "/content/drive/MyDrive/CrashGeoNN/" + animation_name
animate_simulation_vtk(train_static[sid],model,X_DYNAMIC_INDEX, x_scaler,delta_scaler,edge_scaler, edge_geom_scaler, pos_scaler,device="cuda",save_path=animation_path,
    max_edges=4000,stride=1,framerate=15,window_size=(1280, 720),show_points=True,point_size=3.0,show_bc=True,bc_point_size=9.0,
    tube_lines=False,          # True = tubos 3D (más bonito, algo más lento)line_width=1.0,
    camera="iso")


Creating  test animation...


Creating  train animation...


'/content/drive/MyDrive/CrashGeoNN/rollout_train_one_cycle_24.mp4'

In [ ]:
== Mejores parámetros ==
{'weight_decay': 2.3465254539958182e-05, 'layers': 6, 'hidden': 256, 'dropout': 0.12675839829004065, 'lam_bc': 3.19352581602606e-05, 'lam_smooth': 0.012753330601120494, 'lam_kin': 0.012981351206458915, 'lam_geo': 0.033372913516991586, 'smoothl1_beta': 1.0, 'edge_dyn_scale': 1.9803683902320412e-05, 'augment_node_features': False, 'pos_scale': 0.03745928442366847, 'use_edge_dyn': True, 'use_world_edges': False, 'r_world': 18, 'use_input_noise': False, 'noise_warmup_epochs': 5, 'vel_from_disp': False, 'disp_weights': 0.8890076506112339, 'vel_weights': 0.7159661412516282, 'lam_vec': 0.5170142513796345, 'lam_norm': 0.05714986186823738, 'lam_ang': 0.24105233141330312, 'p_teacher_p0': 0.9174666557886781, 'p_teacher_pmin': 0.26406754884832057, 'p_teacher_warmup': 12, 'max_grad_norm': 0.8388359885282621, 'k_min_start': 1, 'k_min_end': 2, 'k_max_start': 4, 'k_max_end': 13, 'lr_min': 4.952926387339979e-05, 'lr_max': 0.0010901907508582193}
Mejor FDE: 19.941668290358322
[Top 1] value=19.94167 params={'weight_decay': 2.3465254539958182e-05, 'layers': 6, 'hidden': 256, 'dropout': 0.12675839829004065, 'lam_bc': 3.19352581602606e-05, 'lam_smooth': 0.012753330601120494, 'lam_kin': 0.012981351206458915, 'lam_geo': 0.033372913516991586, 'smoothl1_beta': 1.0, 'edge_dyn_scale': 1.9803683902320412e-05, 'augment_node_features': False, 'pos_scale': 0.03745928442366847, 'use_edge_dyn': True, 'use_world_edges': False, 'r_world': 18, 'use_input_noise': False, 'noise_warmup_epochs': 5, 'vel_from_disp': False, 'disp_weights': 0.8890076506112339, 'vel_weights': 0.7159661412516282, 'lam_vec': 0.5170142513796345, 'lam_norm': 0.05714986186823738, 'lam_ang': 0.24105233141330312, 'p_teacher_p0': 0.9174666557886781, 'p_teacher_pmin': 0.26406754884832057, 'p_teacher_warmup': 12, 'max_grad_norm': 0.8388359885282621, 'k_min_start': 1, 'k_min_end': 2, 'k_max_start': 4, 'k_max_end': 13, 'lr_min': 4.952926387339979e-05, 'lr_max': 0.0010901907508582193}
[Top 2] value=20.94625 params={'weight_decay': 8.00579546377368e-05, 'layers': 6, 'hidden': 256, 'dropout': 0.08366792353025942, 'lam_bc': 6.557443847155155e-05, 'lam_smooth': 0.0017164982874392372, 'lam_kin': 0.027257423271994857, 'lam_geo': 0.020660093144148402, 'smoothl1_beta': 1.0, 'edge_dyn_scale': 1.3937609487932563e-05, 'augment_node_features': False, 'pos_scale': 0.46071307880368073, 'use_edge_dyn': True, 'use_world_edges': False, 'r_world': 18, 'use_input_noise': False, 'noise_warmup_epochs': 5, 'vel_from_disp': False, 'disp_weights': 0.8886673056598304, 'vel_weights': 0.6225747934347707, 'lam_vec': 0.522544341546585, 'lam_norm': 0.18787484817007535, 'lam_ang': 4.5175265364714307e-05, 'p_teacher_p0': 0.8694931598303873, 'p_teacher_pmin': 0.2921757503473813, 'p_teacher_warmup': 14, 'max_grad_norm': 0.8858047337131755, 'k_min_start': 1, 'k_min_end': 2, 'k_max_start': 4, 'k_max_end': 10, 'lr_min': 5.140440966796931e-05, 'lr_max': 0.0011288758619670428}
[Top 3] value=21.05103 params={'weight_decay': 2.417484483195105e-05, 'layers': 6, 'hidden': 256, 'dropout': 0.12826213727719918, 'lam_bc': 3.5291515652378285e-05, 'lam_smooth': 0.01212835888542156, 'lam_kin': 0.017106826539431717, 'lam_geo': 0.01211298630589386, 'smoothl1_beta': 1.0, 'edge_dyn_scale': 1.8836734464476936e-05, 'augment_node_features': False, 'pos_scale': 0.4035742850049058, 'use_edge_dyn': True, 'use_world_edges': False, 'r_world': 16, 'use_input_noise': False, 'noise_warmup_epochs': 5, 'vel_from_disp': False, 'disp_weights': 0.9237265464220773, 'vel_weights': 0.6905145196298735, 'lam_vec': 0.5296655072045322, 'lam_norm': 0.09743523381992669, 'lam_ang': 0.07853162838110934, 'p_teacher_p0': 0.9606450364853606, 'p_teacher_pmin': 0.27776611999888384, 'p_teacher_warmup': 12, 'max_grad_norm': 0.7423392879911914, 'k_min_start': 1, 'k_min_end': 2, 'k_max_start': 4, 'k_max_end': 9, 'lr_min': 4.094521719955534e-05, 'lr_max': 0.0013719378033451182}
[Top 4] value=21.17194 params={'weight_decay': 6.981538666627014e-05, 'layers': 10, 'hidden': 256, 'dropout': 0.12622721942622486, 'lam_bc': 8.547047510430638e-05, 'lam_smooth': 0.0009820499341112877, 'lam_kin': 0.013123722624601547, 'lam_geo': 0.022598070367610523, 'smoothl1_beta': 1.0, 'edge_dyn_scale': 1.0770008868726248e-05, 'augment_node_features': False, 'pos_scale': 0.4575622656476359, 'use_edge_dyn': True, 'use_world_edges': False, 'r_world': 19, 'use_input_noise': False, 'noise_warmup_epochs': 10, 'vel_from_disp': False, 'disp_weights': 0.8352344071281336, 'vel_weights': 0.7074270654382425, 'lam_vec': 0.6559214810920302, 'lam_norm': 0.41391341810803567, 'lam_ang': 0.8440555267623734, 'p_teacher_p0': 0.9634524304711131, 'p_teacher_pmin': 0.28987865185334377, 'p_teacher_warmup': 17, 'max_grad_norm': 0.760982813230394, 'k_min_start': 1, 'k_min_end': 2, 'k_max_start': 3, 'k_max_end': 16, 'lr_min': 0.00010657772321483307, 'lr_max': 0.0008790904412277317}
[Top 5] value=21.81404 params={'weight_decay': 4.2623438743547625e-05, 'layers': 6, 'hidden': 256, 'dropout': 0.0852320186503375, 'lam_bc': 2.177990551337495e-05, 'lam_smooth': 0.0037300113323039375, 'lam_kin': 0.032095478362996505, 'lam_geo': 0.0004555877076854653, 'smoothl1_beta': 1.0, 'edge_dyn_scale': 3.848518277414547e-05, 'augment_node_features': False, 'pos_scale': 0.445089246305818, 'use_edge_dyn': True, 'use_world_edges': False, 'r_world': 16, 'use_input_noise': False, 'noise_warmup_epochs': 15, 'vel_from_disp': False, 'disp_weights': 0.875154680159335, 'vel_weights': 0.6275721519301223, 'lam_vec': 0.524622041695988, 'lam_norm': 0.17860099226945064, 'lam_ang': 0.2862848674849449, 'p_teacher_p0': 0.8147033893609568, 'p_teacher_pmin': 0.28375274630051106, 'p_teacher_warmup': 15, 'max_grad_norm': 0.7114295567085798, 'k_min_start': 1, 'k_min_end': 2, 'k_max_start': 4, 'k_max_end': 15, 'lr_min': 4.0097130740537577e-05, 'lr_max': 0.0012084260367708783}

SyntaxError: invalid syntax (ipython-input-2819920018.py, line 1)